# Is a 1960s adapter all you need? — Phase-1 experiment
**Closed-form RLS correction on a frozen forecaster vs. gradient-trained adapters, under a leakage-free online protocol with delayed feedback.**

This notebook implements the Week-1/2 core of the plan and the **go/no-go gate**:

- A frozen backbone (DLinear, closed-form ridge; optional PatchTST via torch) is pre-trained on the first part of the series.
- The rest of the series is streamed. At each origin `t` the model sees the lookback `y[t-L:t]` and predicts `y[t:t+H]`.
- **Delay parameter `d`** (in stream steps): the label window of origin `s` may be used for adapter updates only once the current origin `i >= s + d`.
  - `d = 1` → **leaky** protocol (FSNet/OneNet-style; label windows overlap unobserved future). Included *only* as a reference to quantify inflation.
  - `d = H` → honest minimum (Proceed/DSOF setting: a label matures only after the full horizon has been observed).
  - `d = 2H, 4H` → extra feedback lag (realistic pipelines).
- Adapters compared on **identical cached backbone forecasts**: closed-form **RLS** (calibration + feature variants, one forgetting factor, no learning rate) vs. **NLMS**, **Bias-EMA**, and **gradient-trained adapters of the same capacity** (SGD across a learning-rate grid + Adam) as the in-notebook proxy for learned TTA modules. TAFAS/COSA/PETSA plug in later through the cached-forecast interface (last section).

**Claims ladder evaluated at the end**
- **Rung A** — RLS matches/beats the best gradient adapter (oracle-selected lr — generous to GD) at honest `d = H`.
- **Rung B** — RLS Pareto-dominates on tuning-freeness/compute at a small accuracy cost.
- **Rung C** — the RLS advantage grows with `d`.

**v2 additions (Aug 19):** NLMS mu-grid and decayed-lr GD (fairness), extended lambda sweep, **RLS-Bank** (follow-the-leader over forgetting factors — fully tuning-free and delay-adaptive), lambda(d)/staleness and per-horizon diagnostics, moving-block-bootstrap CIs, and a multi-seed CUDA-ready PatchTST backbone.

Everything is cached under an experiment folder: raw data, standardization stats, backbone weights, frozen online forecasts, per-run results (keyed by config hash), figures. Re-running the notebook skips all completed work.

In [ ]:
import os, json, time, hashlib, urllib.request, warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.random.seed(0)

PROTO_V = 3   # bump to invalidate old results after protocol/feature changes
SMOKE = os.environ.get("SMOKE", "0") == "1"      # tiny run to validate the pipeline
RUN_FULL = os.environ.get("RUN_FULL", "0") == "1"  # adds ETTm1 + d=4H

@dataclass
class Config:
    exp_root: str = "rls_delay_tta"
    datasets: tuple = ("ETTh2", "ETTm1", "weather", "electricity", "traffic")  # FULL EDITION: auto-skips missing CSVs
    lookback: int = 96
    horizon: int = 24
    train_frac: float = 0.25          # backbone pre-training region
    val_frac: float = 0.05            # ridge-alpha selection
    delays: tuple = (1, 24, 48) if SMOKE else ((1, 24, 48, 96) if RUN_FULL else (1, 24, 48, 96))
    online_cap: int = 4000 if SMOKE else 0    # 0 = full online stream
    # DLinear (closed-form)
    ma_kernel: int = 25
    ridge_alphas: tuple = (1e-3, 1e-1, 1e1)
    # adapters
    rls_lambda: float = 0.999           # effective memory ~1/(1-lam) samples
    rls_lambda_sweep: tuple = (0.99, 0.995, 0.999, 1.0)
    rls_p0: float = 10.0                # prior gain on standardized data
    rls_trace_clip: float = 10.0        # cap trace(P)/F at p0 -> bounded gain (anti covariance-windup)
    nlms_mu: float = 0.1
    nlms_leak: float = 1e-3             # leaky-NLMS pull toward pass-through (textbook nonstationary variant)
    bias_beta: float = 0.05
    gd_lrs: tuple = (1e-4, 1e-3, 1e-2, 1e-1)
    adam_lr: float = 1e-2
    gd_decay_lrs: tuple = (1e-3, 1e-2, 1e-1)   # SGD with lr_t = lr0/sqrt(1+t/tau)
    gd_decay_tau: float = 1000.0
    nlms_mus: tuple = (0.02, 0.05, 0.3)        # grid around the 0.1 default (fairness)
    bank_lams: tuple = (1.0, 0.9999, 0.9995, 0.999, 0.995, 0.99)  # RLS-Bank experts (desc: ties -> safest)
    bank_gamma: float = 0.99            # discount on the selection ledger (memory ~100 matured samples)
    rls_lambda_sweep_ext: tuple = (0.9995, 0.9999)
    # bootstrap CIs
    boot_B: int = 500
    boot_block_mult: int = 2                   # block length = mult * seasonal period
    # review round: closest-relative baseline (ELF/AdapTS-style OLS-on-lookback corrector) and trace-clip ablation
    elf_datasets: tuple = ("ETTh2", "ETTm1", "weather")   # O(C*H*L^2) per step: keep to C<=21 streams
    elf_lambda: float = 1.0               # growing-window OLS, as in the released ELF forecaster
    elf_weighter_beta: float = 0.99       # EW-MSE memory of the ELF-style online weighter
    wide_channels: int = 50               # streams wider than this: 1 PatchTST seed and a 6-method external replay (compute budget)
    # PatchTST
    pt_seeds: tuple = (0, 1, 2)
    pt_epochs: int = 20
    pt_d_model: int = 128
    pt_layers: int = 3
    pt_heads: int = 8
    pt_patch: int = 16
    pt_stride: int = 8
    pt_lr: float = 1e-4
    pt_bs: int = 128
    seed: int = 0
    run_patchtst: bool = True         # FULL EDITION: PatchTST on by default (auto-skips if torch missing)

CFG = Config()
PT_SMOKE = os.environ.get("RUN_PT_SMOKE", "0") == "1"      # tiny CPU validation of the PatchTST path
if os.environ.get("RUN_PT", "0") == "1" or PT_SMOKE:
    CFG.run_patchtst = True
if PT_SMOKE:
    CFG.pt_seeds, CFG.pt_epochs, CFG.pt_d_model, CFG.pt_layers, CFG.pt_heads = (0,), 2, 32, 2, 4
SEASONAL_PERIOD = {"ETTh2": 24, "ETTm1": 96, "weather": 144, "electricity": 24, "traffic": 24}  # steps/day
DATA_URLS = {
    "ETTh2": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh2.csv",
    "ETTm1": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv",
    "weather": None, "electricity": None, "traffic": None,   # Drive-hosted: drop the Autoformer/TSLib CSVs into data/
}
CSV_NAME = {ds: f"{ds}.csv" for ds in DATA_URLS}

ROOT = Path(CFG.exp_root)
DIRS = {name: ROOT / name for name in ["data", "backbones", "forecasts", "results", "figures"]}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)
(ROOT / "config_snapshot.json").write_text(json.dumps(asdict(CFG), indent=2, default=str))
print(f"Experiment folder: {ROOT.resolve()}  |  SMOKE={SMOKE}  RUN_FULL={RUN_FULL}")
print(json.dumps(asdict(CFG), indent=2, default=str))

## Caching layer
Every expensive artifact is keyed by a SHA-1 hash of its exact configuration and stored inside the experiment folder. `run_id` collisions are impossible across different settings, and re-executing any cell first checks the cache. `manifest.csv` accumulates one row per completed run.

In [ ]:
def cfg_hash(d: dict) -> str:
    return hashlib.sha1(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:12]

def result_path(run_id):
    return DIRS["results"] / f"{run_id}.json"

def load_result(run_id):
    p = result_path(run_id)
    if not p.exists():
        return None
    if not (DIRS["results"] / f"{run_id}_curves.npz").exists():
        return None   # orphaned json (interrupted run or partial cache copy) -> treat as not done, recompute
    return json.loads(p.read_text())

def save_result(run_id, payload, curves=None):
    if curves is not None:
        np.savez_compressed(DIRS["results"] / f"{run_id}_curves.npz", **curves)
    result_path(run_id).write_text(json.dumps(payload, indent=2))   # written last: acts as the completion marker
    row = {k: payload.get(k) for k in ["run_id","dataset","backbone","method","delay","mse","mae","sec_per_1k_steps","n_online"]}
    mpath = ROOT / "manifest.csv"
    dfrow = pd.DataFrame([row])
    if mpath.exists():
        m = pd.read_csv(mpath)
        m = m[m.run_id != run_id]
        m = pd.concat([m, dfrow], ignore_index=True)
    else:
        m = dfrow
    m.to_csv(mpath, index=False)

def cached_npz(path: Path, builder):
    """Load an .npz if present, else build via builder() -> dict of arrays, save, return."""
    if path.exists():
        with np.load(path, allow_pickle=False) as z:
            return {k: z[k] for k in z.files}
    out = builder()
    np.savez_compressed(path, **out)
    return out

## Data: why ETTh2 (primary) and ETTm1 (secondary)
Chosen deliberately, not by convenience alone:
1. **They are the shared currency of the literature we are auditing** — FSNet, OneNet (incl. its delayed-feedback appendix), DSOF, Proceed, TAFAS and PETSA all report on ETT, so our numbers are directly interpretable against theirs.
2. **ETTh2 is the drift-heavy one**: its online region undergoes a well-documented level/variance shift (visible below), which is exactly the regime where adaptation should matter — a fair arena for both sides.
3. **ETTm1** adds a second frequency (15-min) and a different drift profile, guarding against single-dataset conclusions.
4. Practical: hosted on the official `zhouhaoyi/ETDataset` GitHub repo → scriptable, cacheable download. Weather/ECL/Traffic (Google-Drive-hosted) are follow-ups on a workstation.

Split: first 25% backbone pre-training, next 5% validation (ridge-α selection), remaining 70% streamed online. Standardization uses **train-region statistics only**.

In [ ]:
def load_dataset(name):
    csv = DIRS["data"] / CSV_NAME[name]
    if not csv.exists():
        print(f"downloading {name} ...")
        urllib.request.urlretrieve(DATA_URLS[name], csv)
    df = pd.read_csv(csv)
    y = df.drop(columns=["date"]).to_numpy(dtype=np.float32)
    n = len(y)
    n_tr = int(n * CFG.train_frac); n_va = int(n * CFG.val_frac)
    mu, sd = y[:n_tr].mean(0), y[:n_tr].std(0) + 1e-8
    z = (y - mu) / sd
    return {"z": z.astype(np.float32), "n_train": n_tr, "n_val": n_va,
            "channels": list(df.columns[1:]), "mu": mu, "sd": sd}

# ---- review round: fetch the Autoformer/TSLib benchmark CSVs that are not on GitHub (weather / electricity / traffic)
_need = [ds for ds in CFG.datasets if DATA_URLS.get(ds) is None and not (DIRS["data"] / CSV_NAME[ds]).exists()]
if _need:
    print("CSVs not found locally:", _need, "- trying the public Autoformer dataset folder ...")
    try:
        import gdown, glob, shutil
        _dl = DIRS["data"] / "_autoformer_release"; _dl.mkdir(exist_ok=True)
        gdown.download_folder(id="1ZOYpTUa82_jCcxIdTmyr0LXQfvaM9vIy", output=str(_dl), quiet=True, use_cookies=False)
        for ds in _need:
            hits = glob.glob(str(_dl / "**" / CSV_NAME[ds]), recursive=True)
            if hits: shutil.copy(hits[0], DIRS["data"] / CSV_NAME[ds]); print("  fetched", CSV_NAME[ds])
    except Exception as e:
        print(f"  automatic download unavailable ({type(e).__name__}: {str(e)[:80]}) -> pip install gdown, "
              f"or copy the CSVs from the Time-Series-Library dataset release into {DIRS['data']}")
_missing = [ds for ds in CFG.datasets if DATA_URLS.get(ds) is None and not (DIRS["data"] / CSV_NAME[ds]).exists()]
if _missing:
    print(f"skipping (CSV not found in {DIRS['data']}): {_missing} — drop weather.csv / electricity.csv / traffic.csv")
    print("from the standard Autoformer / thuml Time-Series-Library dataset release there to include them.")
CFG.datasets = tuple(ds for ds in CFG.datasets if ds not in _missing)
DATA = {ds: load_dataset(ds) for ds in CFG.datasets}
for ds in CFG.datasets:
    C_ = DATA[ds]["z"].shape[1]
    if C_ > 50:
        print(f"NOTE {ds}: C={C_} channels — the adapter grid on this stream is CPU-heavy (expect hours, cached incrementally).")
for ds, D in DATA.items():
    print(f"{ds}: N={len(D['z'])}, C={D['z'].shape[1]}, train={D['n_train']}, val={D['n_val']}, online={len(D['z'])-D['n_train']-D['n_val']}")

In [ ]:
# Drift visualization: rolling mean/std per channel with split boundaries
fig, axes = plt.subplots(len(CFG.datasets), 2, figsize=(12, 3.2*len(CFG.datasets)), squeeze=False)
for r, ds in enumerate(CFG.datasets):
    D = DATA[ds]; z = D["z"]; w = SEASONAL_PERIOD[ds]*7
    roll = pd.DataFrame(z).rolling(w, min_periods=w)
    for ax, stat, ttl in [(axes[r,0], roll.mean(), "rolling mean (1w)"), (axes[r,1], roll.std(), "rolling std (1w)")]:
        ax.plot(stat.to_numpy(), lw=0.6, alpha=0.8)
        for x0 in (D["n_train"], D["n_train"]+D["n_val"]):
            ax.axvline(x0, color="k", ls="--", lw=1)
        ax.set_title(f"{ds} — {ttl} (dashed: train|val|online)"); ax.set_xlabel("t")
fig.tight_layout(); fig.savefig(DIRS["figures"]/"drift_overview.png", dpi=120); plt.show()

## Frozen backbone — DLinear via closed-form ridge
DLinear = moving-average trend/seasonal decomposition + one linear map `L→H` per component, per channel. Because it is linear, the SGD training used in the literature merely approximates the ridge solution — so we solve it **exactly** (per channel, per component, with an intercept), selecting the ridge α on the validation region. Deterministic, seconds to fit, cached.

Decomposition is **window-local with replicate padding** (exactly as in the official DLinear code) — no centered smoothing over the full series, which would leak future values into the trend.

In [ ]:
def decompose(W, k):
    """W: [n, L, C] lookback windows -> (trend, seasonal), window-local MA with replicate padding."""
    pf, pb = (k-1)//2, k-1-(k-1)//2
    P = np.concatenate([np.repeat(W[:, :1], pf, 1), W, np.repeat(W[:, -1:], pb, 1)], axis=1)
    cs = np.cumsum(P, axis=1, dtype=np.float64)
    cs = np.concatenate([np.zeros_like(cs[:, :1]), cs], axis=1)
    trend = ((cs[:, k:] - cs[:, :-k]) / k).astype(np.float32)
    return trend, W - trend

def make_windows(z, origins, L, H):
    sw = np.lib.stride_tricks.sliding_window_view(z, L, axis=0)      # [N-L+1, C, L]
    X = sw[origins - L].transpose(0, 2, 1)                            # [n, L, C]
    swy = np.lib.stride_tricks.sliding_window_view(z, H, axis=0)
    Y = swy[origins].transpose(0, 2, 1)                               # [n, H, C]
    return X, Y

def _aug(M):  # add intercept column
    return np.concatenate([M, np.ones_like(M[:, :1])], axis=1)

def fit_dlinear(z, n_train, n_val, L, H, k, alphas):
    tr = np.arange(L, n_train - H)
    va = np.arange(n_train, n_train + n_val - H)
    Xtr, Ytr = make_windows(z, tr, L, H); Xva, Yva = make_windows(z, va, L, H)
    Ttr, Str = decompose(Xtr, k); Tva, Sva = decompose(Xva, k)
    C = z.shape[1]; best = None
    for a in alphas:
        Wt = np.zeros((C, L+1, H), np.float32); Ws = np.zeros_like(Wt)
        for c in range(C):
            for comp, Wm in ((Ttr[:, :, c], Wt), (Str[:, :, c], Ws)):
                A = _aug(comp)
                G = A.T @ A + a * np.eye(L+1, dtype=np.float32)
                Wm[c] = np.linalg.solve(G, A.T @ Ytr[:, :, c])
        pv = np.stack([_aug(Tva[:, :, c]) @ Wt[c] + _aug(Sva[:, :, c]) @ Ws[c] for c in range(C)], -1)
        mse = float(np.mean((pv - Yva) ** 2))
        if best is None or mse < best[0]:
            best = (mse, a, Wt, Ws)
    return {"Wt": best[2], "Ws": best[3], "alpha": np.float32(best[1]), "val_mse": np.float32(best[0])}

def dlinear_predict(z, origins, model, L, H, k):
    X, Y = make_windows(z, origins, L, H)
    T, S = decompose(X, k); C = z.shape[1]
    P = np.stack([_aug(T[:, :, c]) @ model["Wt"][c] + _aug(S[:, :, c]) @ model["Ws"][c] for c in range(C)], -1)
    return P.astype(np.float32), Y   # [n, H, C] each

BACKBONES = {}
for ds in CFG.datasets:
    D = DATA[ds]
    key = cfg_hash({"ds": ds, "bb": "dlinear", "L": CFG.lookback, "H": CFG.horizon,
                    "k": CFG.ma_kernel, "alphas": CFG.ridge_alphas, "tf": CFG.train_frac, "vf": CFG.val_frac})
    path = DIRS["backbones"] / f"dlinear_{ds}_{key}.npz"
    t0 = time.perf_counter()
    BACKBONES[(ds, "dlinear", 0)] = cached_npz(path, lambda D=D: fit_dlinear(
        D["z"], D["n_train"], D["n_val"], CFG.lookback, CFG.horizon, CFG.ma_kernel, CFG.ridge_alphas))
    m = BACKBONES[(ds, "dlinear", 0)]
    print(f"{ds}/dlinear: alpha={float(m['alpha']):g}, val_mse={float(m['val_mse']):.4f} "
          f"({'cache' if time.perf_counter()-t0 < 0.5 else f'{time.perf_counter()-t0:.1f}s'})  -> {path.name}")

### Optional second backbone — PatchTST (requires `torch`)
Off by default (`CFG.run_patchtst=False`) so the notebook runs on any machine. On a GPU box, set it to `True`; weights and forecasts cache exactly like DLinear. The audit's claims should ultimately be shown on a non-linear backbone too — a linear backbone plus a linear adapter is the friendliest case for RLS, and reviewers will ask.

In [ ]:
PATCHTST_AVAILABLE = False
if CFG.run_patchtst:
    try:
        import torch, torch.nn as nn
        PATCHTST_AVAILABLE = True
        DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        print("torch", torch.__version__, "on", DEVICE)
    except ImportError:
        print("torch not installed -> skipping PatchTST (pip install torch)")
if PATCHTST_AVAILABLE:
    class PatchTST(nn.Module):
        def __init__(self, L, H, C, patch, stride, d, heads, layers):
            super().__init__()
            self.patch, self.stride = patch, stride
            n_p = (L - patch)//stride + 1
            self.embed = nn.Linear(patch, d)
            self.pos = nn.Parameter(torch.zeros(1, n_p, d))
            enc = nn.TransformerEncoderLayer(d, heads, 4*d, dropout=0.1, batch_first=True, norm_first=True)
            self.enc = nn.TransformerEncoder(enc, layers)
            self.head = nn.Linear(n_p*d, H)
        def forward(self, x):                      # [B, L, C] -> [B, H, C] channel-independent
            B, L, C = x.shape
            xc = x.permute(0, 2, 1).reshape(B*C, L)
            p = xc.unfold(-1, self.patch, self.stride)
            h = self.enc(self.embed(p) + self.pos).flatten(1)
            return self.head(h).reshape(B, C, -1).permute(0, 2, 1)

    def train_patchtst(ds, seed):
        torch.manual_seed(seed); np.random.seed(seed)
        D = DATA[ds]; z = D["z"]; L, H = CFG.lookback, CFG.horizon
        key = cfg_hash({"ds": ds, "bb": "patchtst", "L": L, "H": H, "ep": CFG.pt_epochs, "lr": CFG.pt_lr,
                        "d": CFG.pt_d_model, "lay": CFG.pt_layers, "hd": CFG.pt_heads,
                        "p": CFG.pt_patch, "s": CFG.pt_stride, "seed": seed})
        ckpt = DIRS["backbones"] / f"patchtst_{ds}_s{seed}_{key}.pt"
        model = PatchTST(L, H, z.shape[1], CFG.pt_patch, CFG.pt_stride, CFG.pt_d_model, CFG.pt_heads, CFG.pt_layers).to(DEVICE)
        if ckpt.exists():
            model.load_state_dict(torch.load(ckpt, map_location=DEVICE)); return model.eval()
        tr = np.arange(L, D["n_train"] - H); va = np.arange(D["n_train"], D["n_train"] + D["n_val"] - H)
        Xtr, Ytr = make_windows(z, tr, L, H); Xva, Yva = make_windows(z, va, L, H)
        Xva_t, Yva_t = torch.from_numpy(Xva).to(DEVICE), torch.from_numpy(Yva).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=CFG.pt_lr); best, bad = 1e9, 0
        # channel-independent batching multiplies the effective batch by C; PyTorch's fused attention refuses
        # B*C > 65535 in training mode, so cap the window batch on wide streams (Traffic: 76 windows -> 65,512 sequences)
        bs = min(CFG.pt_bs, max(8, 65535 // z.shape[1]))
        if bs != CFG.pt_bs: print(f"  {ds}: training batch capped at {bs} windows ({bs * z.shape[1]} channel-sequences)")
        for ep in range(CFG.pt_epochs):
            model.train(); perm = np.random.permutation(len(Xtr))
            for i in range(0, len(perm), bs):
                idx = perm[i:i + bs]
                xb = torch.from_numpy(Xtr[idx]).to(DEVICE); yb = torch.from_numpy(Ytr[idx]).to(DEVICE)
                loss = nn.functional.mse_loss(model(xb), yb)
                opt.zero_grad(); loss.backward(); opt.step()
            model.eval()
            with torch.no_grad():
                vm = float(nn.functional.mse_loss(model(Xva_t), Yva_t))
            print(f"  {ds}/patchtst seed{seed} ep{ep} val={vm:.4f}")
            if vm < best - 1e-5:
                best, bad = vm, 0; torch.save(model.state_dict(), ckpt)
            else:
                bad += 1
                if bad >= 3: break
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE)); return model.eval()

    for ds in CFG.datasets:
        seeds = CFG.pt_seeds if DATA[ds]["z"].shape[1] <= CFG.wide_channels else CFG.pt_seeds[:1]
        for sd in seeds:
            BACKBONES[(ds, "patchtst", sd)] = train_patchtst(ds, sd)


## Frozen online forecasts (cached once, replayed by every adapter)
The backbone is frozen, so its forecasts over the online stream are computed **once** per (dataset, backbone) and cached. Every adapter then replays from the identical cached tensor — this makes method sweeps near-instant, guarantees all adapters see exactly the same inputs, and is the interface external TTA methods will plug into.

Tensors (all `[T, C, H]` or `[T, C]`, float32): `preds` (backbone), `trues`, `last` (last observed value at each origin), `mday` (mean of the last seasonal period), `snv` (seasonal-naive values).

In [ ]:
def gen_forecasts(ds, bb, seed=0):
    D = DATA[ds]; z = D["z"]; L, H = CFG.lookback, CFG.horizon
    P = SEASONAL_PERIOD[ds]
    start = D["n_train"] + D["n_val"]; end = len(z) - H
    origins = np.arange(start, end)
    if CFG.online_cap: origins = origins[:CFG.online_cap]
    if bb == "dlinear":
        pr, tr = dlinear_predict(z, origins, BACKBONES[(ds, bb, seed)], L, H, CFG.ma_kernel)
    else:
        import torch
        model = BACKBONES[(ds, bb, seed)]; X, tr = make_windows(z, origins, L, H); outs = []
        with torch.no_grad():
            for i in range(0, len(X), 512):
                xb = torch.from_numpy(X[i:i+512]).to(DEVICE)
                outs.append(model(xb).cpu().numpy())
        pr = np.concatenate(outs)
    swH = np.lib.stride_tricks.sliding_window_view(z, H, axis=0)   # [*, C, H]
    swP = np.lib.stride_tricks.sliding_window_view(z, P, axis=0)
    return {"origins": origins.astype(np.int64),
            "preds": pr.transpose(0, 2, 1).copy(),        # [T, C, H]
            "trues": tr.transpose(0, 2, 1).copy(),
            "last":  z[origins - 1].copy(),               # [T, C]
            "mday":  swP[origins - P].mean(-1).astype(np.float32),
            "snv":   swH[origins - P].copy()}             # [T, C, H]

def BB_VER(bb):
    if bb == "dlinear": return None
    return f"pt_d{CFG.pt_d_model}l{CFG.pt_layers}h{CFG.pt_heads}p{CFG.pt_patch}s{CFG.pt_stride}ep{CFG.pt_epochs}lr{CFG.pt_lr:g}"

FC = {}
for (ds, bb, seed) in BACKBONES:
    hcfg = {"ds": ds, "bb": bb, "L": CFG.lookback, "H": CFG.horizon,
            "cap": CFG.online_cap, "tf": CFG.train_frac, "vf": CFG.val_frac}
    if seed: hcfg["seed"] = seed        # seed=0 omitted so v1 dlinear caches stay valid
    if BB_VER(bb): hcfg["bbv"] = BB_VER(bb)
    key = cfg_hash(hcfg)
    tag = f"{ds}_{bb}" + (f"_s{seed}" if seed else "")
    FC[(ds, bb, seed)] = cached_npz(DIRS["forecasts"] / f"{tag}_{key}.npz",
                                    lambda ds=ds, bb=bb, seed=seed: gen_forecasts(ds, bb, seed))
    f = FC[(ds, bb, seed)]
    mse_bb = float(np.mean((f["preds"] - f["trues"])**2))
    print(f"{ds}/{bb}/s{seed}: T={len(f['origins'])} online origins, frozen-backbone online MSE={mse_bb:.4f}")
PAIRS = sorted({(k[0], k[1]) for k in FC})

def build_feats(f, featset):
    T, C, H = f["preds"].shape
    ones = np.ones((T, C, H, 1), np.float32)
    base = f["preds"][..., None]
    if featset == "calib":
        return np.concatenate([base, ones], -1)                                   # F=2
    dev_r = (f["last"] - f["mday"])[:, :, None, None]        # recent deviation from daily level
    dev_s = (f["snv"] - f["mday"][:, :, None])[..., None]     # seasonal-shape deviation
    ext = [base, ones, np.broadcast_to(dev_r, (T, C, H, 1)), dev_s]
    return np.concatenate(ext, -1).astype(np.float32)                             # F=4 (de-correlated)

## Adapters
All adapters are linear maps on the same features, one independent unit per (channel, horizon-step), initialized as pass-through — so at step 0 every method equals the frozen backbone, and any difference is attributable to the *update rule alone*:

| method | update rule | hyperparameters |
|---|---|---|
| `frozen` | none | — |
| `bias_ema` | EMA of residual | β |
| `nlms_feat` | normalized LMS (1960) | μ (scale-free) |
| `rls_*` | **closed-form recursive least squares** | forgetting λ only |
| `gd_*_sgd` / `gd_feat_adam` | gradient step on squared error — the stand-in for gradient-trained TTA adapters | learning rate (grid) |

`*_calib` uses features `[ŷ, 1]` (scale-and-shift recalibration, the COSA-style output-space correction); `*_feat` adds `[last - daymean, seasonalnaive - daymean]` deviation features (F=4, de-correlated to avoid RLS covariance wind-up), giving the adapter a chance to model residual structure.

In [ ]:
class Adapter:
    name = "frozen"; featset = "calib"
    def __init__(self, C, H, F): self.w = np.zeros((C, H, F), np.float32); self.w[..., 0] = 1.0
    def predict(self, x): return np.einsum("chf,chf->ch", self.w, x)
    def update(self, x, y): pass

class BiasEMA(Adapter):
    def __init__(self, C, H, F, beta): super().__init__(C, H, F); self.b = np.zeros((C, H), np.float32); self.beta = beta
    def predict(self, x): return x[..., 0] + self.b
    def update(self, x, y): self.b += self.beta * ((y - x[..., 0]) - self.b)

class NLMS(Adapter):
    def __init__(self, C, H, F, mu, leak): 
        super().__init__(C, H, F); self.mu, self.leak = mu, leak; self.w0 = self.w.copy()
    def update(self, x, y):
        e = y - np.einsum("chf,chf->ch", self.w, x)
        self.w += self.mu * e[..., None] * x / (1e-6 + (x * x).sum(-1, keepdims=True))
        self.w -= self.leak * (self.w - self.w0)

class RLS(Adapter):
    def __init__(self, C, H, F, lam, p0, clip):
        super().__init__(C, H, F); self.lam = lam; self.clip = clip
        self.P = np.broadcast_to(np.eye(F, dtype=np.float32) * p0, (C, H, F, F)).copy()
    def update(self, x, y):
        Px = np.einsum("chij,chj->chi", self.P, x)
        kg = Px / (self.lam + np.einsum("chf,chf->ch", x, Px))[..., None]
        e = y - np.einsum("chf,chf->ch", self.w, x)
        self.w += kg * e[..., None]
        self.P = (self.P - kg[..., :, None] * Px[..., None, :]) / self.lam
        self.P = 0.5 * (self.P + self.P.transpose(0, 1, 3, 2))
        # anti-windup: bound the average eigenvalue of P (standard covariance limiting)
        F = self.P.shape[-1]
        tr = np.einsum("chff->ch", self.P) / F
        scale = np.minimum(1.0, self.clip / np.maximum(tr, 1e-12))
        self.P *= scale[..., None, None]
        bad = ~(np.isfinite(self.w).all(-1) & np.isfinite(tr))
        if bad.any():
            self.w[bad] = 0; self.w[bad, ..., 0] = 1.0
            self.P[bad] = np.eye(F, dtype=np.float32) * self.clip

class SGD(Adapter):
    def __init__(self, C, H, F, lr): super().__init__(C, H, F); self.lr = lr
    def update(self, x, y):
        e = y - np.einsum("chf,chf->ch", self.w, x)
        self.w += self.lr * e[..., None] * x

class AdamA(Adapter):
    def __init__(self, C, H, F, lr): 
        super().__init__(C, H, F); self.lr = lr; self.t = 0
        self.m = np.zeros_like(self.w); self.v = np.zeros_like(self.w)
    def update(self, x, y):
        e = y - np.einsum("chf,chf->ch", self.w, x)
        g = -e[..., None] * x; self.t += 1
        self.m = 0.9*self.m + 0.1*g; self.v = 0.999*self.v + 0.001*g*g
        mh = self.m/(1-0.9**self.t); vh = self.v/(1-0.999**self.t)
        self.w -= self.lr * mh/(np.sqrt(vh)+1e-8)

class RLSBankMed(Adapter):
    """Proposed bank: passthrough expert + K RLS experts (one per forgetting factor); prediction is the
    per-(channel,horizon) MEDIAN of expert predictions. No selection ledger at all: hard selection under
    d-step feedback lag rides an exploding expert for up to d steps per episode (see the FTL/discounted-FTL
    ablations, which fail for exactly this reason), while the median is bounded by the sane majority at
    every step regardless of lag. Closed-form, zero hyperparameters beyond the lambda grid itself."""
    def __init__(self, C, H, F, lams, p0, clip):
        super().__init__(C, H, F)
        self.lams = np.array(sorted(lams, reverse=True), np.float32)
        K = len(self.lams) + 1
        self.w = np.zeros((K, C, H, F), np.float32); self.w[..., 0] = 1.0   # expert 0 = frozen passthrough
        self.P = np.broadcast_to(np.eye(F, dtype=np.float32) * p0, (K-1, C, H, F, F)).copy()
        self.clip = clip
    def predict(self, x):
        return np.median(np.einsum("kchf,chf->kch", self.w, x), axis=0)
    def update(self, x, y):
        wr = self.w[1:]
        e = y[None] - np.einsum("kchf,chf->kch", wr, x)
        lam = self.lams[:, None, None]
        Px = np.einsum("kchij,chj->kchi", self.P, x)
        kg = Px / (lam + np.einsum("kchf,chf->kch", Px, x))[..., None]
        wr += kg * e[..., None]
        self.P = (self.P - kg[..., :, None] * Px[..., None, :]) / lam[..., None, None]
        self.P = 0.5 * (self.P + self.P.transpose(0, 1, 2, 4, 3))
        F = self.P.shape[-1]
        tr = np.einsum("kchff->kch", self.P) / F
        self.P *= np.minimum(1.0, self.clip / np.maximum(tr, 1e-12))[..., None, None]
        bad = ~(np.isfinite(wr).all(-1) & np.isfinite(tr))
        if bad.any():
            wr[bad] = 0.0
            wr[..., 0][bad] = 1.0
            self.P[bad] = np.eye(F, dtype=np.float32) * self.clip

class RLSBankFTL(Adapter):
    """ABLATION: undiscounted follow-the-leader over RLS forgetting factors, no passthrough expert.
    Kept because its failure (ledger never forgets -> locks onto pre-shift winners) motivates the
    discounted bank below."""
    def __init__(self, C, H, F, lams, p0, clip):
        super().__init__(C, H, F)
        self.lams = np.array(sorted(lams, reverse=True), np.float32)
        K = len(self.lams)
        self.w = np.zeros((K, C, H, F), np.float32); self.w[..., 0] = 1.0
        self.P = np.broadcast_to(np.eye(F, dtype=np.float32) * p0, (K, C, H, F, F)).copy()
        self.clip = clip
        self.cum = np.zeros((K, C, H), np.float32)
        self.sel = np.zeros((C, H), np.int64)
    def predict(self, x):
        pk = np.einsum("kchf,chf->kch", self.w, x)
        return np.take_along_axis(pk, self.sel[None], 0)[0]
    def update(self, x, y):
        pk = np.einsum("kchf,chf->kch", self.w, x)
        e = y[None] - pk
        self.cum += e * e
        self.sel = np.argmin(self.cum, 0)
        lam = self.lams[:, None, None]
        Px = np.einsum("kchij,chj->kchi", self.P, x)
        kg = Px / (lam + np.einsum("kchf,chf->kch", Px, x))[..., None]
        self.w += kg * e[..., None]
        self.P = (self.P - kg[..., :, None] * Px[..., None, :]) / lam[..., None, None]
        self.P = 0.5 * (self.P + self.P.transpose(0, 1, 2, 4, 3))
        F = self.P.shape[-1]
        tr = np.einsum("kchff->kch", self.P) / F
        self.P *= np.minimum(1.0, self.clip / np.maximum(tr, 1e-12))[..., None, None]
        bad = ~(np.isfinite(self.w).all(-1) & np.isfinite(tr))
        if bad.any():
            self.w[bad] = 0.0
            self.w[..., 0][bad] = 1.0
            self.P[bad] = np.eye(F, dtype=np.float32) * self.clip

class SGDDecay(Adapter):
    """SGD with lr_t = lr0/sqrt(1+t/tau) — mirrors the decayed/adaptive schedules used by gradient TTA."""
    def __init__(self, C, H, F, lr0, tau): super().__init__(C, H, F); self.lr0, self.tau, self.t = lr0, tau, 0
    def update(self, x, y):
        self.t += 1
        e = y - np.einsum("chf,chf->ch", self.w, x)
        self.w += (self.lr0 / np.sqrt(1.0 + self.t / self.tau)) * e[..., None] * x

class RLSBank(Adapter):
    """Passthrough expert + K RLS experts (one per forgetting factor), selected per (channel, horizon) by
    follow-the-DISCOUNTED-leader on matured a-priori errors. Plain FTL fails here (cumulative loss never
    forgets, so it locks onto experts that were good before a regime shift — we keep that ablation in the
    tables); discounting the ledger (gamma) restores adaptivity. Expert 0 is frozen passthrough, so with all
    ledgers tied at the start the bank behaves exactly like the frozen backbone until evidence accrues, and
    it can locally choose "do not adapt" whenever adaptation hurts. Closed-form; no learning rate anywhere."""
    def __init__(self, C, H, F, lams, p0, clip, gamma):
        super().__init__(C, H, F)
        self.lams = np.array(sorted(lams, reverse=True), np.float32)
        K = len(self.lams) + 1                                # +1 passthrough expert at index 0
        self.w = np.zeros((K, C, H, F), np.float32); self.w[..., 0] = 1.0
        self.P = np.broadcast_to(np.eye(F, dtype=np.float32) * p0, (K-1, C, H, F, F)).copy()
        self.clip, self.gamma = clip, gamma
        self.cum = np.zeros((K, C, H), np.float32)
        self.sel = np.zeros((C, H), np.int64)                 # start on passthrough
    def predict(self, x):
        pk = np.einsum("kchf,chf->kch", self.w, x)
        return np.take_along_axis(pk, self.sel[None], 0)[0]
    def update(self, x, y):
        pk = np.einsum("kchf,chf->kch", self.w, x)            # a-priori predictions, every expert
        e = y[None] - pk
        self.cum = self.gamma * self.cum + e * e              # discounted ledger
        self.sel = np.argmin(self.cum, 0)                     # ties -> lowest index = safest
        lam = self.lams[:, None, None]
        wr, er = self.w[1:], e[1:]                            # only RLS experts update
        Px = np.einsum("kchij,chj->kchi", self.P, x)
        kg = Px / (lam + np.einsum("kchf,chf->kch", Px, x))[..., None]
        wr += kg * er[..., None]
        self.P = (self.P - kg[..., :, None] * Px[..., None, :]) / lam[..., None, None]
        self.P = 0.5 * (self.P + self.P.transpose(0, 1, 2, 4, 3))
        F = self.P.shape[-1]
        tr = np.einsum("kchff->kch", self.P) / F
        self.P *= np.minimum(1.0, self.clip / np.maximum(tr, 1e-12))[..., None, None]
        bad = ~(np.isfinite(wr).all(-1) & np.isfinite(tr))
        if bad.any():
            wr[bad] = 0.0
            wr[..., 0][bad] = 1.0
            self.P[bad] = np.eye(F, dtype=np.float32) * self.clip


## Delay-aware online replay
At stream step `i` (origin index), the label of origin `j = i - d` is *released* and consumed by one adapter update, **then** the prediction for origin `i` is emitted. With `d ≥ H` the released label window `y[j : j+H]` lies entirely in the observed past — leakage-free by construction. `d = 1` reproduces the leaky protocol (label windows extend into the unobserved future) and is reported only as a reference.

In [ ]:
def replay(f, adapter, delay):
    X = build_feats(f, adapter.featset)
    T = X.shape[0]; preds = np.empty_like(f["preds"]); trues = f["trues"]
    t0 = time.perf_counter()
    for i in range(T):
        j = i - delay
        if j >= 0:
            adapter.update(X[j], trues[j])
        preds[i] = adapter.predict(X[i])
    sec = time.perf_counter() - t0
    err = preds - trues
    per_step = (err ** 2).mean(axis=(1, 2))
    sub = max(1, T // 400)
    return {"mse": float((err**2).mean()), "mae": float(np.abs(err).mean()),
            "by_h": (err**2).mean(axis=(0,1)).round(5).tolist(),
            "sec_per_1k_steps": round(1000*sec/T, 4), "n_online": T,
            "curve": (np.cumsum(per_step)/np.arange(1, T+1))[::sub].astype(np.float32),
            "curve_x": np.arange(T)[::sub]}

def naive_metrics(f, kind):
    T, C, H = f["preds"].shape
    p = np.broadcast_to(f["last"][:, :, None], (T, C, H)) if kind == "naive" else f["snv"]
    err = p - f["trues"]; per_step = (err**2).mean(axis=(1,2)); sub = max(1, T//400)
    return {"mse": float((err**2).mean()), "mae": float(np.abs(err).mean()),
            "by_h": (err**2).mean(axis=(0,1)).round(5).tolist(),
            "sec_per_1k_steps": 0.0, "n_online": T,
            "curve": (np.cumsum(per_step)/np.arange(1, T+1))[::sub].astype(np.float32),
            "curve_x": np.arange(T)[::sub]}

## Run grid (fully cached — re-running skips completed runs)

In [ ]:
def method_zoo(C, H):
    zoo = []
    def A(name, family, featset, ctor):
        def make():
            F = 2 if featset == "calib" else 4
            a = ctor(C, H, F); a.name, a.featset = name, featset; return a
        zoo.append((name, family, make))
    A("frozen", "frozen", "calib", lambda C,H,F: Adapter(C,H,F))
    A("bias_ema", "simple", "calib", lambda C,H,F: BiasEMA(C,H,F, CFG.bias_beta))
    A("nlms_feat", "classic", "feat", lambda C,H,F: NLMS(C,H,F, CFG.nlms_mu, CFG.nlms_leak))
    for mu in CFG.nlms_mus:
        A(f"nlms_feat_mu{mu:g}", "classic", "feat", lambda C,H,F,m=mu: NLMS(C,H,F, m, CFG.nlms_leak))
    A("rls_calib", "rls", "calib", lambda C,H,F: RLS(C,H,F, CFG.rls_lambda, CFG.rls_p0, CFG.rls_trace_clip))
    A("rls_feat", "rls", "feat", lambda C,H,F: RLS(C,H,F, CFG.rls_lambda, CFG.rls_p0, CFG.rls_trace_clip))
    A("rls_bank_ftl", "rls_ablation", "feat", lambda C,H,F: RLSBankFTL(C,H,F, CFG.bank_lams, CFG.rls_p0, CFG.rls_trace_clip))
    A("rls_bank_dftl", "rls_ablation", "feat", lambda C,H,F: RLSBank(C,H,F, CFG.bank_lams, CFG.rls_p0, CFG.rls_trace_clip, CFG.bank_gamma))
    A("rls_bank_med", "rls", "feat", lambda C,H,F: RLSBankMed(C,H,F, CFG.bank_lams, CFG.rls_p0, CFG.rls_trace_clip))
    A("rls_feat_noclip", "rls_ablation", "feat", lambda C,H,F: RLS(C,H,F, CFG.rls_lambda, CFG.rls_p0, 1e12))       # trace-clip ablation (clip disabled)
    A("rls_bank_med_noclip", "rls_ablation", "feat", lambda C,H,F: RLSBankMed(C,H,F, CFG.bank_lams, CFG.rls_p0, 1e12))
    for lam in tuple(CFG.rls_lambda_sweep) + tuple(CFG.rls_lambda_sweep_ext):
        if abs(lam - CFG.rls_lambda) > 1e-12:
            A(f"rls_feat_l{lam:g}", "rls_sweep", "feat", lambda C,H,F,l=lam: RLS(C,H,F, l, CFG.rls_p0, CFG.rls_trace_clip))
    for lr in CFG.gd_lrs:
        A(f"gd_calib_sgd{lr:g}", "gd", "calib", lambda C,H,F,l=lr: SGD(C,H,F, l))
        A(f"gd_feat_sgd{lr:g}",  "gd", "feat",  lambda C,H,F,l=lr: SGD(C,H,F, l))
    for lr in CFG.gd_decay_lrs:
        A(f"gd_feat_sgddec{lr:g}", "gd", "feat", lambda C,H,F,l=lr: SGDDecay(C,H,F, l, CFG.gd_decay_tau))
    A(f"gd_feat_adam{CFG.adam_lr:g}", "gd", "feat", lambda C,H,F: AdamA(C,H,F, CFG.adam_lr))
    return zoo

done, ran = 0, 0
for (ds, bb, seed), f in FC.items():
    T, C, H = f["preds"].shape
    for delay in CFG.delays:
        for kind in ("naive", "snaive"):
            hcfg = {"ds": ds, "bb": bb, "m": kind, "d": int(delay), "cap": CFG.online_cap, "L": CFG.lookback, "H": H, "v": PROTO_V}
            if seed: hcfg["seed"] = seed
            if BB_VER(bb): hcfg["bbv"] = BB_VER(bb)
            rid = cfg_hash(hcfg)
            if load_result(rid): done += 1; continue
            r = naive_metrics(f, kind); curve = {"curve": r.pop("curve"), "curve_x": r.pop("curve_x")}
            save_result(rid, {"run_id": rid, "dataset": ds, "backbone": bb, "method": kind, "proto_v": PROTO_V,
                              "family": "naive", "delay": int(delay), "seed": seed, "bbv": BB_VER(bb), **r}, curve); ran += 1
        for name, family, make in method_zoo(C, H):
            hcfg = {"ds": ds, "bb": bb, "m": name, "d": int(delay), "cap": CFG.online_cap,
                    "L": CFG.lookback, "H": H, "p0": CFG.rls_p0, "v": PROTO_V}
            if seed: hcfg["seed"] = seed
            if BB_VER(bb): hcfg["bbv"] = BB_VER(bb)
            rid = cfg_hash(hcfg)
            if load_result(rid): done += 1; continue
            r = replay(f, make(), int(delay)); curve = {"curve": r.pop("curve"), "curve_x": r.pop("curve_x")}
            save_result(rid, {"run_id": rid, "dataset": ds, "backbone": bb, "method": name, "proto_v": PROTO_V,
                              "family": family, "delay": int(delay), "seed": seed, "bbv": BB_VER(bb), **r}, curve); ran += 1
            if ran % 25 == 0: print(f"  ...{ran} runs, latest {ds}/{bb}/s{seed} d={delay} {name} mse={r['mse']:.4f}")
print(f"grid complete: {ran} runs executed, {done} loaded from cache")

## Review round: ELF/AdapTS-style corrector (closest modern relative) as a run baseline
Our re-implementation of the released ELF recipe in the harness's own terms: a per-(channel, horizon) linear forecaster
fit by recursive least squares on the raw lookback window (growing-window OLS at $\lambda{=}1$, the form the ELF paper
calls OLS-equivalent), combined with the frozen forecast by an online inverse-error weighter. Both the fit and the
weighter update **only on released labels**, so it lives on the same delay axis as everything else. `elf_linear` is the
linear forecaster alone; `elf_mix` is the weighted combination. Restricted to streams with $C\le 21$ (cost is $O(CHL^2)$ per step).

In [ ]:
class ELFStyle:
    def __init__(self, C, H, L, lam, beta, p0):
        self.L, self.lam, self.beta = L, lam, beta
        F = L + 1
        self.w = np.zeros((C, H, F), np.float64)
        self.P = np.broadcast_to(np.eye(F) * p0, (C, H, F, F)).copy()
        self.e_fm = np.ones((C, H)); self.e_lin = np.ones((C, H)) * 10.0   # linear starts distrusted
    def feats(self, z, o):                                                  # lookback features of origin o -> [C, L+1]
        return np.concatenate([z[o - self.L:o].T.astype(np.float64), np.ones((z.shape[1], 1))], 1)
    def predict_lin(self, x):                                               # [C, F] -> [C, H]
        return np.einsum("chf,cf->ch", self.w, x)
    def mix_weight(self):                                                   # weight on the linear forecaster
        return self.e_fm / (self.e_fm + self.e_lin + 1e-12)
    def update(self, x, y, fm_pred):
        lin = self.predict_lin(x)
        self.e_fm = self.beta * self.e_fm + (1 - self.beta) * (fm_pred - y) ** 2
        self.e_lin = self.beta * self.e_lin + (1 - self.beta) * (lin - y) ** 2
        Px = np.einsum("chij,cj->chi", self.P, x)
        kg = Px / (self.lam + np.einsum("cf,chf->ch", x, Px))[..., None]
        self.w += kg * (y - lin)[..., None]
        self.P = (self.P - kg[..., :, None] * Px[..., None, :]) / self.lam
        self.P = 0.5 * (self.P + self.P.transpose(0, 1, 3, 2))

def _metrics(preds, trues, sec):
    T = len(preds); err = preds - trues; per_step = (err ** 2).mean(axis=(1, 2)); sub = max(1, T // 400)
    return {"mse": float((err**2).mean()), "mae": float(np.abs(err).mean()),
            "by_h": (err**2).mean(axis=(0, 1)).round(5).tolist(), "sec_per_1k_steps": round(1000*sec/T, 4), "n_online": T,
            "curve": (np.cumsum(per_step)/np.arange(1, T+1))[::sub].astype(np.float32), "curve_x": np.arange(T)[::sub]}

def replay_elf(f, z, delay):
    T, C, H = f["preds"].shape; orig = f["origins"]; trues = f["trues"]
    a = ELFStyle(C, H, CFG.lookback, CFG.elf_lambda, CFG.elf_weighter_beta, CFG.rls_p0)
    pl = np.empty_like(f["preds"]); pm = np.empty_like(f["preds"]); t0 = time.perf_counter()
    for i in range(T):
        j = i - delay
        if j >= 0:
            a.update(a.feats(z, orig[j]), trues[j].astype(np.float64), f["preds"][j].astype(np.float64))
        lin = a.predict_lin(a.feats(z, orig[i])); wl = a.mix_weight()
        pl[i] = lin; pm[i] = wl * lin + (1 - wl) * f["preds"][i]
    sec = time.perf_counter() - t0
    return {"elf_linear": _metrics(pl, trues, sec), "elf_mix": _metrics(pm, trues, sec)}

def run_elf_rows(f, z, ds, bb, seed, key_extra):
    bbv = bb if str(bb).endswith("_ext") else BB_VER(bb)
    for delay in CFG.delays:
        rids = {m: cfg_hash({"elf": m, "ds": ds, "bb": bb, "seed": seed, "d": int(delay), "lam": CFG.elf_lambda,
                             "beta": CFG.elf_weighter_beta, "L": CFG.lookback, "cap": CFG.online_cap, "v": PROTO_V, **key_extra})
                for m in ("elf_linear", "elf_mix")}
        if all(load_result(r) for r in rids.values()):
            continue
        out = replay_elf(f, z, int(delay))
        for m, r in out.items():
            curve = {"curve": r.pop("curve"), "curve_x": r.pop("curve_x")}
            save_result(rids[m], {"run_id": rids[m], "dataset": ds, "backbone": bb, "method": m, "proto_v": PROTO_V,
                                  "family": "elf", "delay": int(delay), "seed": seed, "bbv": bbv, **r}, curve)
            print(f"  elf {ds}/{bb}/s{seed} d={delay} {m:<11} mse={r['mse']:.4f}")

for (ds, bb, seed), f in FC.items():
    if ds in CFG.elf_datasets:
        run_elf_rows(f, DATA[ds]["z"], ds, bb, seed, {})
print("ELF-style baseline complete")

## Results

In [ ]:
results = [json.loads(p.read_text()) for p in DIRS["results"].glob("*.json")
           if (DIRS["results"] / (p.stem + "_curves.npz")).exists()]
R = pd.DataFrame(results)
if "seed" not in R.columns: R["seed"] = 0
R["seed"] = R["seed"].fillna(0).astype(int)
if "bbv" not in R.columns: R["bbv"] = None
R["bbv"] = R["bbv"].astype(object).where(pd.notna(R["bbv"]), None)   # missing/NaN -> None (pre-bbv rows are dlinear)
validT = {(k[0], k[1]): v["preds"].shape[0] for k, v in FC.items()}
def _keep_row(r):
    if str(r.backbone).endswith("_ext"):
        return abs(validT.get((r.dataset, "dlinear"), -9) - r.n_online) <= 2 and r.get("proto_v") == PROTO_V
    return validT.get((r.dataset, r.backbone)) == r.n_online and r.get("proto_v") == PROTO_V and r["bbv"] == BB_VER(r.backbone)
R = R[R.apply(_keep_row, axis=1)]
# seed-mean aggregate used by tables/plots/verdict (pivot_table also averages seeds implicitly)
RA = R.assign(mse=R.mse.fillna(np.inf)).groupby(["dataset","backbone","family","method","delay"], as_index=False).agg(mse=("mse","mean"), mse_std=("mse","std"), n_seeds=("seed","nunique"))
PAIRS_T = sorted(set(PAIRS) | {(r.dataset, r.backbone) for _, r in RA.iterrows() if str(r.backbone).endswith("_ext")})
for (ds, bb) in PAIRS_T:
    sub = RA[(RA.dataset == ds) & (RA.backbone == bb)]
    if not len(sub): continue
    tab = sub.pivot_table(index=["family", "method"], columns="delay", values="mse").round(4)  # inf = diverged
    tab.columns = [f"d={c}" + (" (leaky)" if c == 1 else "") for c in tab.columns]
    ns = int(sub.n_seeds.max())
    print(f"=== {ds} / {bb} — online MSE (standardized), mean over {ns} seed(s) ==="); print(tab.to_string()); print()
    tab.to_csv(DIRS["results"] / f"table_{ds}_{bb}.csv")

In [ ]:
HON = CFG.horizon
def best_gd(sub, d):
    g = sub[(sub.family == "gd") & (sub.delay == d) & np.isfinite(sub.mse)]
    return g.loc[g.mse.idxmin()] if len(g) else None

for (ds, bb) in PAIRS:
    sub = RA[(RA.dataset == ds) & (RA.backbone == bb)]
    delays = sorted(sub.delay.unique())
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    # (1) MSE vs delay
    ax = axes[0]
    series = {"frozen": ("frozen", "k--"), "bias_ema": ("bias_ema", "c-"), "nlms_feat": ("nlms_feat", "g-"),
              "rls_calib": ("rls_calib", "C1-"), "rls_feat": ("rls_feat", "r-o")}
    for label, (m, sty) in series.items():
        v = [sub[(sub.method == m) & (sub.delay == d)].mse.mean() for d in delays]
        ax.plot(range(len(delays)), v, sty, label=label)
    ax.plot(range(len(delays)), [best_gd(sub, d).mse for d in delays], "b-s", label="best GD (oracle lr)")
    nv = sub[sub.method == "naive"].mse.mean(); ax.axhline(nv, color="gray", ls=":", label="persistence")
    ax.set_xticks(range(len(delays))); ax.set_xticklabels([f"{d}" + ("\n(leaky)" if d == 1 else "") for d in delays])
    ax.set_xlabel("feedback delay d (steps)"); ax.set_ylabel("online MSE"); ax.legend(fontsize=8); ax.set_title(f"{ds}/{bb}")
    # (2) hyperparameter sensitivity at honest d=H
    ax = axes[1]
    gd = sub[(sub.family == "gd") & (sub.delay == HON) & (sub.method.str.contains("feat"))]
    rl = sub[(sub.family.isin(["rls", "rls_sweep"])) & (sub.delay == HON) & (sub.method.str.contains("feat"))]
    gm = gd.mse[np.isfinite(gd.mse)]
    ax.scatter([0]*len(gm), gm, c="b", label=f"GD-feat over lr grid (spread {gm.max()/gm.min():.2f}x, {int((~np.isfinite(gd.mse)).sum())} diverged)")
    rm = rl.mse[np.isfinite(rl.mse)]
    ax.scatter([1]*len(rm), rm, c="r", label=f"RLS-feat over lambda sweep (spread {rm.max()/rm.min():.2f}x)")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["GD (lr grid)", "RLS (lambda grid)"])
    ax.set_ylabel("online MSE at d=H"); ax.legend(fontsize=8); ax.set_title("hyperparameter sensitivity")
    # (3) cumulative MSE at d=H
    ax = axes[2]
    for m, ccol in [("frozen", "k"), ("rls_feat", "r"), ("rls_bank_med", "m"), (best_gd(sub, HON).method, "b")]:
        rows = R[(R.dataset == ds) & (R.backbone == bb) & (R.method == m) & (R.delay == HON) & (R.seed == 0)]
        if not len(rows): continue
        cpath = DIRS["results"] / f"{rows.iloc[0].run_id}_curves.npz"
        if not cpath.exists(): continue
        z = np.load(cpath)
        ax.plot(z["curve_x"], z["curve"], ccol, lw=1.2, label=m)
    ax.set_xlabel("online step"); ax.set_ylabel("cumulative MSE"); ax.legend(fontsize=8); ax.set_title(f"cumulative error, d=H={HON}")
    fig.tight_layout(); fig.savefig(DIRS["figures"] / f"summary_{ds}_{bb}.png", dpi=130); plt.show()

# compute table
cmp_tab = R[(R.delay == HON) & np.isfinite(R.mse.fillna(np.inf))].groupby("method")["sec_per_1k_steps"].mean().sort_values().round(3)
print("--- adapter cost, seconds per 1k online steps (d=H) ---"); print(cmp_tab.to_string())

## Go / no-go — claims ladder

In [ ]:
verdicts = {}
for (ds, bb) in PAIRS:
    sub = RA[(RA.dataset == ds) & (RA.backbone == bb)]
    rls = sub[(sub.family == "rls") & (sub.delay == HON)].mse.min()          # rls family incl. rls_bank_ftl
    bank = float(sub[(sub.method == "rls_bank_med") & (sub.delay == HON)].mse.iloc[0])
    gdo = best_gd(sub, HON).mse
    frozen = sub[(sub.method == "frozen") & (sub.delay == HON)].mse.iloc[0]
    naive = sub[(sub.method == "naive") & (sub.delay == HON)].mse.iloc[0]
    gd_spread = sub[(sub.family == "gd") & (sub.delay == HON)].mse
    rl_spread = sub[(sub.family.isin(["rls", "rls_sweep"])) & (sub.delay == HON) & (sub.method != "rls_bank_ftl")].mse
    hi = [d for d in sorted(sub.delay.unique()) if d >= HON]
    gap = {int(d): float(best_gd(sub, d).mse - sub[(sub.family == "rls") & (sub.delay == d)].mse.min()) for d in hi}
    gap_bank = {int(d): float(best_gd(sub, d).mse - sub[(sub.method == "rls_bank_med") & (sub.delay == d)].mse.iloc[0]) for d in hi}
    rungA = rls <= gdo * 1.01
    gd_ok = gd_spread[np.isfinite(gd_spread)]; rl_ok = rl_spread[np.isfinite(rl_spread)]
    div_pen = float("inf") if (~np.isfinite(gd_spread)).any() else (gd_ok.max()/gd_ok.min())
    rungB = (rl_ok.max()/rl_ok.min()) < div_pen
    rungC = len(hi) >= 2 and all(gap[hi[k+1]] >= gap[hi[k]] - 1e-6 for k in range(len(hi)-1)) and gap[hi[-1]] > 0
    rungCp = all(v >= -0.01 * gdo for v in gap_bank.values())   # C-prime: tuning-free bank at parity (or better) at EVERY delay
    verdicts[f"{ds}/{bb}"] = {"rls_best": round(float(rls),4), "rls_bank_med": round(bank,4), "gd_oracle": round(float(gdo),4),
        "frozen": round(float(frozen),4), "persistence": round(float(naive),4),
        "gap_gdminus_rls_by_delay": {k: round(v,4) for k,v in gap.items()},
        "gap_gdminus_bank_by_delay": {k: round(v,4) for k,v in gap_bank.items()},
        "rungA_match_or_beat_oracleGD": bool(rungA), "rungB_less_hparam_sensitive": bool(rungB),
        "rungC_gap_grows_with_delay": bool(rungC),
        "rungCprime_bank_parity_at_all_delays": bool(rungCp),
        "GO": bool(rungA or rungB or rungC or rungCp)}
    v = verdicts[f"{ds}/{bb}"]
    print(f"=== {ds}/{bb} @ honest d=H ===")
    print(f"  best RLS {v['rls_best']} (bank: {v['rls_bank_med']}) | oracle-GD {v['gd_oracle']} | frozen {v['frozen']} | persistence {v['persistence']}")
    print(f"  rung A (match/beat oracle GD): {v['rungA_match_or_beat_oracleGD']}")
    print(f"  rung B (lower hparam sensitivity): {v['rungB_less_hparam_sensitive']}")
    print(f"  rung C (gap grows with delay): {v['rungC_gap_grows_with_delay']}  gaps={v['gap_gdminus_rls_by_delay']}")
    print(f"  rung C' (tuning-free bank >= oracle-GD at every delay): {v['rungCprime_bank_parity_at_all_delays']}  gaps={v['gap_gdminus_bank_by_delay']}")
    print(f"  --> {'GO: at least one rung holds' if v['GO'] else 'NO-GO on this cell'}")
(ROOT / "verdict.json").write_text(json.dumps(verdicts, indent=2))
print("\nCaveats: (1) GD here is a same-capacity proxy, not TAFAS/COSA — repo integration is step 2 and the honest headline; "
      "(2) linear backbone + linear adapter is the friendliest case for RLS — confirm on PatchTST; "
      "(3) if SMOKE=1, the stream is capped — full-stream numbers are the ones that count.")

## Diagnostics I — staleness vs. plasticity, and the fate of rung C
Rung C as originally posed (fixed-lambda RLS advantage grows with delay) failed: as labels get staler, the *optimal plasticity drops* — the winning GD learning rate shrinks with `d`, and the winning RLS forgetting factor rises with `d`. The question that matters for the paper is therefore **rung C-prime**: does the tuning-free RLS-Bank — median over {passthrough, RLS(lambda grid)} experts; the FTL and discounted-FTL selection ablations both fail because hard selection under d-step lag rides exploding experts — track the oracle at every delay without being told the delay? Below: oracle-GD vs oracle-lambda RLS vs the bank across `d`, the optimal-hyperparameter trajectories, and the per-horizon breakdown that explains the persistence result.

In [ ]:
lam_of = lambda m: float(m.split("_l")[1]) if "_l" in m else CFG.rls_lambda
fig, axes = plt.subplots(len(PAIRS), 3, figsize=(15, 4*len(PAIRS)), squeeze=False)
for r, (ds, bb) in enumerate(PAIRS):
    sub = RA[(RA.dataset == ds) & (RA.backbone == bb)]
    hi = [d for d in sorted(sub.delay.unique()) if d >= HON]
    rows = []
    for d in hi:
        g = best_gd(sub, d)
        rl = sub[(sub.family.isin(["rls","rls_sweep"])) & (sub.delay == d) & (~sub.method.str.contains("bank")) & np.isfinite(sub.mse)]
        ro = rl.loc[rl.mse.idxmin()]
        bank = float(sub[(sub.method == "rls_bank_med") & (sub.delay == d)].mse.iloc[0])
        rows.append({"delay": d, "oracle_gd": g.mse, "gd_name": g.method,
                     "oracle_rls": ro.mse, "rls_lam": lam_of(ro.method), "bank": bank,
                     "frozen": float(sub[(sub.method=="frozen") & (sub.delay==d)].mse.iloc[0]),
                     "persistence": float(sub[(sub.method=="naive") & (sub.delay==d)].mse.iloc[0])})
    Dg = pd.DataFrame(rows)
    print(f"=== {ds}/{bb} — oracle curves vs the tuning-free bank ==="); print(Dg.round(4).to_string(index=False)); print()
    ax = axes[r,0]
    for col, sty in [("oracle_gd","b-s"),("oracle_rls","r-o"),("bank","m-^"),("frozen","k--"),("persistence","gray")]:
        ax.plot(Dg.delay, Dg[col], sty if col!="persistence" else ":", color=None if col!="persistence" else "gray", label=col)
    ax.set_xlabel("delay d"); ax.set_ylabel("online MSE"); ax.legend(fontsize=8); ax.set_title(f"{ds}/{bb}: oracles vs bank")
    ax = axes[r,1]
    mem = [min(1.0/(1.0-l), 1e5) if l < 1 else 1e5 for l in Dg.rls_lam]
    lr  = [float(n.split("sgd")[-1].replace("dec","")) if "sgd" in n else np.nan for n in Dg.gd_name]
    ax.plot(Dg.delay, mem, "r-o", label="oracle RLS memory 1/(1-lam)")
    ax2 = ax.twinx(); ax2.plot(Dg.delay, lr, "b-s", label="oracle GD lr"); ax2.set_yscale("log"); ax.set_yscale("log")
    ax.set_xlabel("delay d"); ax.set_ylabel("memory (samples)"); ax2.set_ylabel("lr")
    ax.set_title("optimal plasticity vs staleness"); ax.legend(fontsize=8, loc="upper left"); ax2.legend(fontsize=8, loc="lower right")
    ax = axes[r,2]
    for m, c in [("naive","gray"),("frozen","k"),("rls_feat","r"),("rls_bank_med","m"),(best_gd(sub, HON).method,"b")]:
        row = R[(R.dataset==ds)&(R.backbone==bb)&(R.method==m)&(R.delay==HON)&(R.seed==0)]
        if len(row): ax.plot(range(1, len(row.iloc[0].by_h)+1), row.iloc[0].by_h, c, lw=1.2, label=m)
    ax.set_xlabel("horizon step h"); ax.set_ylabel("MSE"); ax.legend(fontsize=7); ax.set_title(f"per-horizon at d=H")
fig.tight_layout(); fig.savefig(DIRS["figures"]/"staleness_diagnostics.png", dpi=130); plt.show()

## Diagnostics II — significance via moving-block bootstrap
Everything in the pipeline is deterministic given the backbone, so seed variance is not the uncertainty that matters; temporal dependence is. We compare methods by their **paired per-step loss differentials** and bootstrap the mean with circular moving blocks (block length = 2 seasonal periods, so daily structure survives resampling). Caveat: adjacent forecast origins share window content, so dependence is strong and this CI is indicative, not exact — fine for a go/no-go, and the paper should report it alongside effect sizes.

In [ ]:
import hashlib as _hl
def elf_series(f, z, delay):
    """Per-step MSE of the ELF-style corrector (same loop as replay_elf; returns both variants)."""
    T, C, H = f["preds"].shape; orig = f["origins"]; trues = f["trues"]
    a = ELFStyle(C, H, CFG.lookback, CFG.elf_lambda, CFG.elf_weighter_beta, CFG.rls_p0)
    sl = np.empty(T, np.float32); sm = np.empty(T, np.float32)
    for i in range(T):
        j = i - delay
        if j >= 0:
            a.update(a.feats(z, orig[j]), trues[j].astype(np.float64), f["preds"][j].astype(np.float64))
        lin = a.predict_lin(a.feats(z, orig[i])); wl = a.mix_weight(); mix = wl * lin + (1 - wl) * f["preds"][i]
        sl[i] = ((lin - trues[i]) ** 2).mean(); sm[i] = ((mix - trues[i]) ** 2).mean()
    return {"elf_linear": sl, "elf_mix": sm}

SERIES_DIR = DIRS["results"] / "series"; SERIES_DIR.mkdir(exist_ok=True)
_ELF_MEMO = {}
def per_step_mse(f, method_name, delay, C, H, ds=None, key=None):
    """Per-step MSE series; disk-cached under results/series/ when a cache key is given (one replay per series, ever)."""
    p = (SERIES_DIR / f"{cfg_hash({**key, 'm': method_name, 'd': int(delay)})}.npy") if key is not None else None
    if p is not None and p.exists():
        return np.load(p)
    if method_name in ("elf_linear", "elf_mix"):
        assert ds is not None and ds in DATA
        mk = (json.dumps(key, sort_keys=True) if key is not None else id(f), int(delay))
        if mk not in _ELF_MEMO:
            _ELF_MEMO[mk] = elf_series(f, DATA[ds]["z"], delay)   # one loop serves both variants
            if key is not None:                                   # and both land on disk immediately
                for vn, arr in _ELF_MEMO[mk].items():
                    np.save(SERIES_DIR / f"{cfg_hash({**key, 'm': vn, 'd': int(delay)})}.npy", arr)
        return _ELF_MEMO[mk][method_name]
    if method_name == "naive":
        pn = np.broadcast_to(f["last"][:, :, None], f["preds"].shape)
        out = ((pn - f["trues"])**2).mean(axis=(1,2))
        if p is not None: np.save(p, out)
        return out
    make = dict((n, mk) for n, _fam, mk in method_zoo(C, H))[method_name]
    X = build_feats(f, make().featset); a = make()
    T = X.shape[0]; out = np.empty(T, np.float32)
    for i in range(T):
        j = i - delay
        if j >= 0: a.update(X[j], f["trues"][j])
        out[i] = float((((a.predict(X[i])) - f["trues"][i])**2).mean())
    if p is not None: np.save(p, out)
    return out

def mbb_ci(diff, l, B, rng):
    T = len(diff); ext = np.concatenate([diff, diff[:l]])
    bm = np.convolve(ext, np.ones(l)/l, mode="valid")[:T]      # circular block means
    k = int(np.ceil(T / l))
    draws = bm[rng.integers(0, T, size=(B, k))].mean(1)
    return float(diff.mean()), float(np.quantile(draws, 0.025)), float(np.quantile(draws, 0.975))

ci_path = ROOT / "bootstrap_ci.json"
ci_rows = json.loads(ci_path.read_text()) if ci_path.exists() else []
# cache key includes stream length and backbone version: a SMOKE (capped) run must never satisfy a full run
have = {(r["dataset"], r["backbone"], r["delay"], r["pair"], r.get("n_online"), r.get("bbv")) for r in ci_rows}
rng = np.random.default_rng(0)
for (ds, bb) in PAIRS:
    f = FC[(ds, bb, 0)]; T, C, H = f["preds"].shape; bbv = BB_VER(bb)
    sub = RA[(RA.dataset == ds) & (RA.backbone == bb)]
    wide = C > CFG.wide_channels                     # wide streams: essential pairs at d=H only (compute budget)
    for d in ([HON] if wide else [HON, max(CFG.delays)]):
        gname = best_gd(sub, d).method
        series = {}
        pairs = [("rls_bank_med", gname), ("rls_bank_med", "frozen"), ("rls_bank_med", "naive")] + ([] if wide else [("rls_feat", gname)])
        if ds in CFG.elf_datasets:                       # review round: the ELF-style corrector against bank, oracle-GD, persistence
            pairs += [("elf_mix", "rls_bank_med"), ("elf_linear", "rls_bank_med"), ("elf_mix", gname), ("elf_mix", "naive"), ("elf_linear", "naive")]
        skey = {"ds": ds, "bb": bb, "seed": 0, "T": int(T), "bbv": bbv, "v": PROTO_V}
        for pair in pairs:
            keyt = (ds, bb, int(d), f"{pair[0]} - {pair[1]}", T, bbv)
            if keyt in have: continue
            print(f"  CI {ds}/{bb} d={d}: {pair[0]} - {pair[1]} ...", flush=True)
            for m in pair:
                if m not in series: series[m] = per_step_mse(f, m, int(d), C, H, ds=ds, key=skey)
            mean, lo, hipp = mbb_ci(series[pair[0]] - series[pair[1]], CFG.boot_block_mult*SEASONAL_PERIOD[ds], CFG.boot_B, rng)
            ci_rows.append({"dataset": ds, "backbone": bb, "delay": int(d), "pair": keyt[3],
                            "n_online": int(T), "bbv": bbv,
                            "mean_diff": round(mean,4), "ci_lo": round(lo,4), "ci_hi": round(hipp,4),
                            "significant": bool(lo > 0 or hipp < 0)})
            have.add(keyt); ci_path.write_text(json.dumps(ci_rows, indent=2))   # incremental: an interrupt loses nothing

CI = pd.DataFrame(ci_rows)
print("negative mean_diff = first method better; * = 95% CI excludes 0")
print(CI.assign(sig=np.where(CI.significant, "*", "")).to_string(index=False))

## Headline: their backbone — their TTA vs our closed-form bank (vs persistence)
Populated once external predictions have been scored. All rows share the identical stream, split and scaling.

In [ ]:
ext = RA[RA.backbone.astype(str).str.endswith("_ext")]
if not len(ext):
    print("no external results yet")
else:
    for (ds, bb) in sorted({(r.dataset, r.backbone) for _, r in ext.iterrows()}):
        sub = RA[(RA.dataset == ds) & (RA.backbone == bb)]
        rows = ["frozen", "tafas", "tafas_tuned", "cosa", "cosa_tuned", "petsa", "dynatta", "rls_bank_med", "rls_feat",
                "rls_bank_med_noclip", "elf_linear", "elf_mix", "gd_feat_sgddec0.001", "gd_feat_sgd0.001", "bias_ema", "naive", "snaive"]
        t = sub[sub.method.isin(rows)].pivot_table(index="method", columns="delay", values="mse").round(4)
        t = t.reindex([r for r in rows if r in t.index])
        print(f"=== {ds} / {bb} — external TTA vs our adapters on the SAME frozen stream ==="); print(t.to_string()); print()
        t.to_csv(DIRS["results"] / f"headline_{ds}_{bb}.csv")

## Plug-in interface for real TTA methods (TAFAS / COSA / PETSA) — step 2
The cell below exports `stream_spec.json` (dataset, split fractions, `L`, `H`, the exact online origin list hash, standardization stats). Run the external repos on **this exact stream** (their own adaptation logic must respect the same release rule: labels of origin `s` usable from step `s+d`), dump their per-origin predictions as an `.npz` with array `preds` of shape `[T, C, H]` on the standardized scale, then call `score_external(path, method_name, delay)` — the result lands in the same manifest/tables as everything above.

**GPU runbook (144 GB box):** `pip install torch` (CUDA wheel), then execute the notebook with `RUN_FULL=1 RUN_PT=1` in the environment — this trains PatchTST x3 seeds on both datasets (early stopping; minutes on your card), regenerates forecasts per seed, and re-runs the full adapter grid on the PatchTST streams; everything lands in the same cache/tables/verdict. `RUN_PT_SMOKE=1` first if you want a 2-epoch sanity pass. Then clone TAFAS (`kimanki/TAFAS`) and COSA (`bigbases/COSA_ICLR2026`), run them on the exported stream specs with the label-release rule enforced inside their update loops, dump `[T,C,H]` prediction npz files, and call `score_external(...)`.

**Next steps per the plan:** (1) gate on the verdict above; (2) if GO — full streams, ETTm1, PatchTST backbone (`CFG.run_patchtst=True` on a GPU box), then TAFAS/COSA via this interface; (3) diagnostic section: gradient staleness vs. drift timescale, per-horizon breakdown (`by_h` is already stored per run); (4) writing.

In [ ]:
for (ds, bb, seed), f in FC.items():
    D = DATA[ds]
    spec = {"dataset": ds, "backbone": bb, "seed": seed, "lookback": CFG.lookback, "horizon": CFG.horizon,
            "train_frac": CFG.train_frac, "val_frac": CFG.val_frac, "online_cap": CFG.online_cap,
            "n_online": int(f["preds"].shape[0]), "origins_sha1": hashlib.sha1(f["origins"].tobytes()).hexdigest()[:12],
            "standardization": {"mu": D["mu"].tolist(), "sd": D["sd"].tolist()},
            "delays": list(map(int, CFG.delays))}
    (ROOT / (f"stream_spec_{ds}_{bb}" + (f"_s{seed}" if seed else "") + ".json")).write_text(json.dumps(spec, indent=2))

def _canonical(ds):
    key = (ds, "dlinear", 0)
    assert key in FC, f"no canonical stream for {ds} — dataset not loaded this session"
    return FC[key]

def _align_to_ours(npz):
    """Intersect the external file's origin grid with ours (the repo's ratio rounding can shift the stream by one origin)."""
    P, their_orig = npz["preds"].astype(np.float32), npz["origins"].astype(np.int64)
    meta = json.loads(str(npz["meta"]))
    f = _canonical(meta["dataset"])
    common, ia, ib = np.intersect1d(f["origins"], their_orig, return_indices=True)
    assert len(common) >= 0.99 * len(f["origins"]), "origin grids do not overlap — check split ratios / seq / pred in the external run"
    if len(common) < len(f["origins"]):
        T0 = len(f["origins"])
        f = {k: (v[ia] if isinstance(v, np.ndarray) and v.ndim >= 1 and v.shape[0] == T0 else v) for k, v in f.items()}
    return P[ib], meta, f

def score_external(npz_path):
    """Ingest an external-method prediction npz (from integration/run_external.py)."""
    z = np.load(npz_path, allow_pickle=False)
    P, meta, f = _align_to_ours(z)
    ds, bb, method = meta["dataset"], meta["backbone"], meta["method"]
    rid = cfg_hash({"ext": True, **{k: meta[k] for k in ("method", "dataset", "backbone", "seed", "delay")},
                    "cap": CFG.online_cap, "v": PROTO_V})
    if load_result(rid):
        return
    err = P - f["trues"]; per_step = (err**2).mean(axis=(1, 2)); sub = max(1, len(P)//400)
    save_result(rid, {"run_id": rid, "dataset": ds, "backbone": bb, "method": method, "proto_v": PROTO_V,
                      "family": "external_tta", "delay": int(meta["delay"]), "seed": int(meta["seed"]), "bbv": bb,
                      "mse": float((err**2).mean()), "mae": float(np.abs(err).mean()),
                      "by_h": (err**2).mean(axis=(0, 1)).round(5).tolist(),
                      "sec_per_1k_steps": float("nan"), "n_online": int(len(P))},
                {"curve": (np.cumsum(per_step)/np.arange(1, len(P)+1))[::sub].astype(np.float32),
                 "curve_x": np.arange(len(P))[::sub]})
    print(f"scored external {method} on {ds}/{bb} d={meta['delay']}: mse={float((err**2).mean()):.4f}")

def replay_zoo_on_external(frozen_npz, methods=None):
    """THEIR frozen backbone + OUR adapters: the apples-to-apples headline on their streams."""
    z = np.load(frozen_npz, allow_pickle=False)
    P, meta, f = _align_to_ours(z)
    assert meta["method"] == "frozen", "pass the frozen__*.npz export"
    ds, bb, seed = meta["dataset"], meta["backbone"], int(meta["seed"])
    f2 = dict(f); f2["preds"] = P
    T, C, H = P.shape
    zoo = {n: mk for n, _fam, mk in method_zoo(C, H)}
    if methods is None:                          # review round: FULL zoo on their frozen stream (= deep cell under their recipe);
        methods = list(zoo) if C <= CFG.wide_channels else [   # essential subset on wide streams (compute budget)
            "frozen", "bias_ema", "rls_feat", "rls_bank_med", "rls_feat_noclip", "rls_bank_med_noclip",
            "gd_feat_sgddec0.001", "gd_feat_sgd0.001", "gd_feat_sgd0.0001"]
    for delay in [d for d in CFG.delays if d >= CFG.horizon]:
        for kind in ("naive", "snaive"):
            rid = cfg_hash({"extzoo": True, "ds": ds, "bb": bb, "m": kind, "d": int(delay), "seed": seed, "v": PROTO_V})
            if not load_result(rid):
                r = naive_metrics(f2, kind); curve = {"curve": r.pop("curve"), "curve_x": r.pop("curve_x")}
                save_result(rid, {"run_id": rid, "dataset": ds, "backbone": bb, "method": kind, "proto_v": PROTO_V,
                                  "family": "naive", "delay": int(delay), "seed": seed, "bbv": bb, **r}, curve)
        for name in methods:
            rid = cfg_hash({"extzoo": True, "ds": ds, "bb": bb, "m": name, "d": int(delay), "seed": seed,
                            "p0": CFG.rls_p0, "v": PROTO_V})
            if load_result(rid):
                continue
            fam = dict((n, f_) for n, f_, _ in method_zoo(C, H))[name]
            r = replay(f2, zoo[name](), int(delay)); curve = {"curve": r.pop("curve"), "curve_x": r.pop("curve_x")}
            save_result(rid, {"run_id": rid, "dataset": ds, "backbone": bb, "method": name, "proto_v": PROTO_V,
                              "family": fam, "delay": int(delay), "seed": seed, "bbv": bb, **r}, curve)
            print(f"  ours-on-theirs {ds}/{bb}/s{seed} d={delay} {name:<22} mse={r['mse']:.4f}")

    if ds in CFG.elf_datasets:
        run_elf_rows(f2, DATA[ds]["z"], ds, bb, seed, {"extzoo": True})

EXT_DIR = ROOT / "external_preds"; EXT_DIR.mkdir(exist_ok=True)
ext_files = sorted(EXT_DIR.glob("*.npz"))
if not ext_files:
    print(f"no external prediction files yet — drop run_external.py outputs into {EXT_DIR}")
for p in ext_files:
    try:
        score_external(p)
    except Exception as e:
        print(f"could not score {p.name}: {e}")
for p in ext_files:
    if p.name.startswith("frozen__"):
        try:
            replay_zoo_on_external(p)
        except Exception as e:
            print(f"could not replay zoo on {p.name}: {e}")

print("stream specs exported:", [p.name for p in ROOT.glob('stream_spec_*.json')])

## Paper figures (publication-ready PDFs)
Regenerates every figure used in the draft directly from the cached results in this folder — including
PatchTST cells and external (their-checkpoint) rows when present — and writes vector PDF + 300-dpi PNG to
`figures/paper/`. Cells degrade gracefully: panels whose data is not in the cache are skipped with a note.
Figure 5 additionally tries to parse COSA's partial-ground-truth counters from the integration `matrix.log`
(path via `MATRIX_LOG` env or the default `../COSA_ICLR2026/matrix.log`), falling back to the values
verified from the recorded run logs.

In [ ]:
import re as _re
import matplotlib as _mpl
PAPER = DIRS["figures"] / "paper"; PAPER.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8,
                     "legend.fontsize": 7, "xtick.labelsize": 7, "ytick.labelsize": 7,
                     "lines.linewidth": 1.2, "figure.dpi": 200, "pdf.fonttype": 42,
                     "axes.spines.top": False, "axes.spines.right": False})
# Okabe-Ito palette (CVD-safe), per the paper style spec; linestyles encode method class / backbone
PCOL = {"persistence": "#999999", "frozen": "#000000", "bias": "#009E73", "bank": "#CC79A7",
        "rls": "#D55E00", "gd": "#0072B2", "tafas": "#E69F00", "cosa": "#56B4E9", "snaive": "#BBBBBB"}
BLS = {"dlinear": "-", "patchtst": "--"}

def _load_frame():
    rows = [json.loads(p.read_text()) for p in DIRS["results"].glob("*.json")
            if (DIRS["results"] / (p.stem + "_curves.npz")).exists()]
    F = pd.DataFrame(rows)
    if "seed" not in F.columns: F["seed"] = 0
    F["seed"] = F["seed"].fillna(0).astype(int)
    if "bbv" not in F.columns: F["bbv"] = None
    F["bbv"] = F["bbv"].astype(object).where(pd.notna(F["bbv"]), None)
    vt = {(k[0], k[1]): v["preds"].shape[0] for k, v in FC.items()}
    def keep(r):
        if str(r.backbone).endswith("_ext"):
            return abs(vt.get((r.dataset, "dlinear"), -9) - r.n_online) <= 2 and r.get("proto_v") == PROTO_V
        return vt.get((r.dataset, r.backbone)) == r.n_online and r.get("proto_v") == PROTO_V and r["bbv"] == BB_VER(r.backbone)
    F = F[F.apply(keep, axis=1)]
    FA = F.assign(mse=F.mse.fillna(np.inf)).groupby(
        ["dataset", "backbone", "family", "method", "delay"], as_index=False).agg(mse=("mse", "mean"))
    return F, FA

PF_R, PF = _load_frame()
P_CELLS = [(ds, bb) for ds in CFG.datasets for bb in ("dlinear", "patchtst")
           if len(PF[(PF.dataset == ds) & (PF.backbone == bb)])]
print("cells available for paper figures:", P_CELLS)
PDEL = sorted(d for d in PF.delay.unique() if d in (1, 24, 48, 96))

def _val(ds, bb, m, d):
    r = PF[(PF.dataset == ds) & (PF.backbone == bb) & (PF.method == m) & (PF.delay == d)].mse
    return float(r.iloc[0]) if len(r) else np.nan

def _bestgd(ds, bb, d):
    g = PF[(PF.dataset == ds) & (PF.backbone == bb) & (PF.family == "gd") & (PF.delay == d)]
    g = g[np.isfinite(g.mse)]
    return (float(g.mse.min()), g.loc[g.mse.idxmin(), "method"]) if len(g) else (np.nan, "")

def _save(fig, name):
    fig.savefig(PAPER / (name + ".pdf"), bbox_inches="tight")
    fig.savefig(PAPER / (name + ".png"), bbox_inches="tight", dpi=300)
    plt.close(fig); print("wrote", name)

# ---- Figure 1: leaky vs honest (ETTh2 / DLinear) ----
if ("ETTh2", "dlinear") in P_CELLS:
    names = ["persistence", "frozen", "bias-EMA", "RLS-Bank", "best GD"]
    cols = [PCOL["persistence"], PCOL["frozen"], PCOL["bias"], PCOL["bank"], PCOL["gd"]]
    def _v5(d):
        return [_val("ETTh2", "dlinear", "naive", d), _val("ETTh2", "dlinear", "frozen", d),
                _val("ETTh2", "dlinear", "bias_ema", d), _val("ETTh2", "dlinear", "rls_bank_med", d),
                _bestgd("ETTh2", "dlinear", d)[0]]
    fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.6), sharey=True)
    for ax, d, ttl in [(axes[0], 1, "leaky protocol ($d{=}1$)"), (axes[1], 24, "honest protocol ($d{=}H$)")]:
        v = _v5(d)
        bars = ax.bar(range(5), v, color=cols, width=0.65)
        ax.axhline(_val("ETTh2", "dlinear", "naive", 24), color=PCOL["persistence"], ls=":", lw=1)
        for b, x in zip(bars, v):
            ax.text(b.get_x() + b.get_width()/2, x + 0.02, f"{x:.2f}", ha="center", fontsize=7.2)
        ax.set_xticks(range(5)); ax.set_xticklabels(names, rotation=20, ha="right"); ax.set_title(ttl)
    axes[0].set_ylabel("online MSE (ETTh2)")
    for ax_, txt_ in [(axes[0], "appears to beat\npersistence"), (axes[1], "+83% vs leaky;\nranking reorders")]:
        ax_.text(2, 0.30, txt_, ha="center", fontsize=7, color=PCOL["bias"],
                 bbox=dict(facecolor="white", edgecolor="none", alpha=0.9, pad=1.5))
    _save(fig, "fig1_leaky_vs_honest")

# ---- Figure 2: protocol schematic ----
fig, ax = plt.subplots(figsize=(6.6, 2.6)); ax.axis("off")
def _tl(y): ax.plot([0.5, 9.5], [y, y], color="0.75", lw=1)
s0, Hh, dd = 2.0, 3.0, 4.4
_tl(2.6)
ax.plot([s0], [2.6], marker="v", color="k"); ax.text(s0, 2.78, "origin $s$", ha="center", fontsize=8)
ax.plot([s0, s0+Hh], [2.45, 2.45], color=PCOL["bank"], lw=5, solid_capstyle="butt")
ax.text(s0+Hh/2, 2.24, "target window $y_{s:s+H}$", ha="center", fontsize=8, color=PCOL["bank"])
ax.plot([s0+Hh], [2.6], marker="^", color=PCOL["bank"]); ax.text(s0+Hh, 2.78, "matured\n$s{+}H$", ha="center", fontsize=7.5)
ax.plot([s0+dd], [2.6], marker="^", color="k"); ax.text(s0+dd, 2.78, "released\n$s{+}d$", ha="center", fontsize=7.5)
ax.annotate("", xy=(s0+dd, 2.06), xytext=(s0+Hh, 2.06), arrowprops=dict(arrowstyle="<->", lw=1))
ax.text((2*s0+Hh+dd)/2, 1.9, "label lag $\ell = d-H$", ha="center", fontsize=8)
_tl(1.3); ax.text(0.4, 1.3, "partial GT\nat lag $\ell$", ha="right", va="center", fontsize=7.5)
ax.plot([s0, s0+1.4], [1.3, 1.3], color=PCOL["bias"], lw=5, solid_capstyle="butt")
ax.plot([s0+1.4, s0+Hh], [1.3, 1.3], color=PCOL["bias"], lw=5, alpha=0.25, solid_capstyle="butt")
ax.text(s0+0.7, 1.1, "usable prefix", ha="center", fontsize=7, color=PCOL["bias"])
ax.text(s0+2.3, 1.1, "not yet a label", ha="center", fontsize=7, color=PCOL["bias"], alpha=0.7)
_tl(0.5); ax.text(0.4, 0.5, "leaky $d{=}1$\n(reference)", ha="right", va="center", fontsize=7.5)
ax.axvspan(6.8, 9.5, ymin=0.02, ymax=0.28, color=PCOL["rls"], alpha=0.08)
ax.text(8.15, 0.72, "unobserved future", ha="center", fontsize=7, color=PCOL["rls"])
ax.annotate("", xy=(8.0, 0.5), xytext=(6.3, 0.5), arrowprops=dict(arrowstyle="->", color=PCOL["rls"], lw=1.4))
ax.text(7.1, 0.32, "updates consume future targets", fontsize=7, color=PCOL["rls"])
ax.set_xlim(0, 10); ax.set_ylim(0.1, 3.0)
_save(fig, "fig2_protocol")

# ---- Figure 3: MSE vs delay across cells ----
if P_CELLS:
    nc = 2; nr = int(np.ceil(len(P_CELLS)/nc))
    fig, axes = plt.subplots(nr, nc, figsize=(6.8, 2.3*nr+0.4), sharex=True, squeeze=False)
    for ax, (ds, bb) in zip(axes.ravel(), P_CELLS):
        x = range(len(PDEL))
        if 1 in PDEL:
            ax.axvspan(PDEL.index(1) - 0.5, PDEL.index(1) + 0.5, color="#DDDDDD", zorder=0)
        ax.plot(x, [_val(ds, bb, "frozen", d) for d in PDEL], "k--", label="frozen")
        ax.plot(x, [_val(ds, bb, "naive", d) for d in PDEL], color=PCOL["persistence"], ls=":", label="persistence")
        ax.plot(x, [_val(ds, bb, "rls_feat", d) for d in PDEL], color=PCOL["rls"], label="RLS ($\lambda{=}0.999$)")
        ax.plot(x, [_val(ds, bb, "rls_bank_med", d) for d in PDEL], color=PCOL["bank"], marker="o", ms=4, label="RLS-Bank")
        ax.plot(x, [_bestgd(ds, bb, d)[0] for d in PDEL], color=PCOL["gd"], marker="s", ms=3.5, label="best GD (oracle lr)")
        ax.set_xticks(list(x)); ax.set_xticklabels([("1\n(leaky)" if d == 1 else str(d)) for d in PDEL])
        ax.set_title(f"{ds} / {'DLinear' if bb=='dlinear' else 'PatchTST (3 seeds)'}")
        if _val(ds, bb, "frozen", 24) > 3: ax.set_yscale("log")
    for ax in axes[:, 0]: ax.set_ylabel("online MSE")
    for ax in axes[-1, :]: ax.set_xlabel("feedback delay $d$ (steps)")
    for ax in axes.ravel()[len(P_CELLS):]: ax.axis("off")
    if 1 in PDEL:
        axes[0, 0].annotate("leaky", xy=(PDEL.index(1), 0.98), xycoords=("data", "axes fraction"),
                            ha="center", va="top", fontsize=6.5, color="#666666")
    _h, _l = axes[0, 0].get_legend_handles_labels()
    fig.legend(_h, _l, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=5, frameon=False)
    fig.tight_layout(); _save(fig, "fig3_mse_vs_delay")

# ---- Figure 4: hyperparameter fragility at d=H ----
if P_CELLS:
    _nr = 2 if len(P_CELLS) > 4 else 1; _nc = int(np.ceil(len(P_CELLS) / _nr))
    fig, axes = plt.subplots(_nr, _nc, figsize=(1.7*_nc, 2.3*_nr), squeeze=False)
    rng = np.random.default_rng(0)
    for ax in axes.ravel()[len(P_CELLS):]: ax.axis("off")
    for ax, (ds, bb) in zip(axes.ravel(), P_CELLS):
        sub = PF_R[(PF_R.dataset == ds) & (PF_R.backbone == bb) & (PF_R.delay == 24)]
        sub = sub.groupby("method", as_index=False).mse.mean()
        gd = sub[sub.method.str.contains("gd_") & sub.method.str.contains("feat")].mse
        rl = sub[sub.method.str.startswith("rls_feat")].mse
        gdf = gd[np.isfinite(gd)]; ndiv = int((~np.isfinite(gd.fillna(np.inf))).sum() + gd.isna().sum())
        ndiv = int((~np.isfinite(gd)).sum())
        ax.scatter(rng.uniform(-0.06, 0.06, len(gdf)), gdf, color=PCOL["gd"], s=16)
        ax.scatter(1 + rng.uniform(-0.06, 0.06, len(rl)), rl, color=PCOL["rls"], s=16)
        ax.set_yscale("log"); ax.set_xticks([0, 1])
        ax.set_xticklabels(["GD\n(lr grid)", "RLS\n($\lambda$ grid)"], fontsize=7.5)
        ax.set_title(f"{ds}/{'DL' if bb=='dlinear' else 'PT'}")
        lo, hi = ax.get_ylim(); ax.set_ylim(lo, hi * 2.2)
        ax.text(0.05, 0.97, f"{ndiv} diverged", transform=ax.transAxes, ha="left", va="top",
                fontsize=6.8, color=PCOL["gd"])
    axes[0, 0].set_ylabel("online MSE at $d{=}H$ (log)")
    fig.tight_layout(); _save(fig, "fig4_fragility")

# ---- Figure 5: COSA delay fragility + its own partial-GT counters ----
EXTF = PF[PF.backbone.astype(str).str.endswith("_ext")]
if len(EXTF):
    mlog = Path(os.environ.get("MATRIX_LOG", Path(CFG.exp_root).resolve().parent / "COSA_ICLR2026" / "matrix.log"))
    counts = {"ETTh2": [487, 0, 0], "ETTm1": [1951, 0, 0]}   # verified from recorded run logs (fallback)
    if mlog.exists():
        try:
            chunks = _re.split(r"RUN\s+", mlog.read_text())
            got = {}
            for ch in chunks:
                m = _re.match(r"cosa__(\w+)__.*__d(\d+)", ch)
                c = _re.findall(r'"partial_gt_adaptation_count":\s*(\d+)', ch)
                if m and c: got[(m.group(1), int(m.group(2)))] = int(c[-1])
            if len(got) >= 4:
                counts = {ds: [got.get((ds, d), 0) for d in (24, 48, 96)] for ds in ("ETTh2", "ETTm1")}
                print("fig5 counters parsed from", mlog)
        except Exception as e:
            print("fig5: matrix.log parse failed, using verified fallback:", e)
    fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.5))
    x = range(3)
    for ds, mk in [("ETTh2", "o"), ("ETTm1", "s")]:
        cv = [_val(ds, "patchtst_ext", "cosa", d) for d in (24, 48, 96)]
        tv = [_val(ds, "patchtst_ext", "tafas", d) for d in (24, 48, 96)]
        if np.isfinite(cv).all(): axes[0].plot(x, cv, color=PCOL["cosa"], marker=mk, label=f"COSA — {ds}")
        if np.isfinite(tv).all(): axes[0].plot(x, tv, color=PCOL["tafas"], marker=mk, ls="--", alpha=0.85, label=f"TAFAS — {ds}")
    axes[0].set_xticks(list(x)); axes[0].set_xticklabels(["24", "48", "96"])
    axes[0].set_xlabel("feedback delay $d$"); axes[0].set_ylabel("online MSE (their checkpoint)")
    axes[0].legend(frameon=False)
    w = 0.35
    axes[1].bar([i - w/2 for i in x], counts["ETTh2"], width=w, color="#444444", label="ETTh2")
    axes[1].bar([i + w/2 for i in x], counts["ETTm1"], width=w, color="#aaaaaa", label="ETTm1")
    for i in list(x)[1:]: axes[1].text(i, max(counts["ETTm1"][0], 1)*0.03, "0", ha="center", fontsize=8)
    axes[1].set_xticks(list(x)); axes[1].set_xticklabels(["$\ell{=}0$", "$\ell{=}24$", "$\ell{=}72$"])
    axes[1].set_ylabel("COSA partial-GT updates\n(its own counters)")
    axes[1].legend(frameon=False, fontsize=7)
    fig.tight_layout(); _save(fig, "fig5_cosa_fragility")
else:
    print("fig5 skipped: no external (their-checkpoint) rows in this cache yet")

# ---- Figure 6: staleness vs plasticity ----
if P_CELLS:
    _dss = [d for d in CFG.datasets if any(p[0] == d for p in P_CELLS)]
    _OI = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#F0E442"]
    dcol = {d: _OI[k % len(_OI)] for k, d in enumerate(_dss)}
    fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.5))
    for (ds, bb) in P_CELLS:
        hi = [d for d in PDEL if d >= HON]
        lrs, mem = [], []
        for d in hi:
            _, gname = _bestgd(ds, bb, d)
            mm = _re.search(r"sgd(?:dec)?([0-9.e-]+)", gname)
            lrs.append(float(mm.group(1)) if mm else np.nan)
            rl = PF[(PF.dataset == ds) & (PF.backbone == bb) & (PF.delay == d)
                    & PF.method.str.startswith("rls_feat") & np.isfinite(PF.mse)]
            lam = CFG.rls_lambda
            if len(rl):
                best = rl.loc[rl.mse.idxmin(), "method"]
                lam = float(best.split("_l")[1]) if "_l" in best else CFG.rls_lambda
            mem.append(min(1.0/(1.0 - lam), 1e5) if lam < 1 else 1e5)
        axes[0].plot(hi, lrs, marker="o", ms=3, color=dcol[ds], ls=BLS.get(bb, "-"))
        axes[1].plot(hi, mem, marker="o", ms=3, color=dcol[ds], ls=BLS.get(bb, "-"))
    for ax, yl in [(axes[0], "oracle GD learning rate"), (axes[1], "oracle RLS memory $1/(1{-}\lambda)$")]:
        ax.set_yscale("log"); ax.set_xlabel("feedback delay $d$"); ax.set_ylabel(yl); ax.set_xticks([24, 48, 96])
    from matplotlib.lines import Line2D
    _hd = [Line2D([0], [0], color=dcol[d], lw=1.6, label=d) for d in _dss] + \
          [Line2D([0], [0], color="#444444", ls=BLS[b], lw=1.4, label=n)
           for b, n in (("dlinear", "DLinear"), ("patchtst", "PatchTST"))]
    axes[0].legend(handles=_hd, frameon=False, ncol=2)
    fig.tight_layout(); _save(fig, "fig6_staleness_plasticity")

# ---- Figures 7-9 (appendix): per-horizon, cumulative, drift ----
if P_CELLS:
    _nr = 2 if len(P_CELLS) > 4 else 1; _nc = int(np.ceil(len(P_CELLS) / _nr))
    fig, axes = plt.subplots(_nr, _nc, figsize=(1.8*_nc, 2.2*_nr), squeeze=False)
    for ax in axes.ravel()[len(P_CELLS):]: ax.axis("off")
    for ax, (ds, bb) in zip(axes.ravel(), P_CELLS):
        for m, c in [("naive", PCOL["persistence"]), ("frozen", "k"), ("rls_feat", PCOL["rls"]),
                     ("rls_bank_med", PCOL["bank"]), (_bestgd(ds, bb, HON)[1], PCOL["gd"])]:
            row = PF_R[(PF_R.dataset == ds) & (PF_R.backbone == bb) & (PF_R.method == m)
                       & (PF_R.delay == HON) & (PF_R.seed == 0)]
            if len(row): ax.plot(range(1, len(row.iloc[0].by_h)+1), row.iloc[0].by_h, color=c, lw=1.1, label=m)
        ax.set_title(f"{ds}/{'DL' if bb=='dlinear' else 'PT'}"); ax.set_xlabel("horizon $h$")
    axes[0, 0].set_ylabel("MSE"); axes[0, 0].legend(frameon=False)
    fig.tight_layout(); _save(fig, "fig7_per_horizon")

    _nr = 2 if len(P_CELLS) > 4 else 1; _nc = int(np.ceil(len(P_CELLS) / _nr))
    fig, axes = plt.subplots(_nr, _nc, figsize=(1.8*_nc, 2.2*_nr), squeeze=False)
    for ax in axes.ravel()[len(P_CELLS):]: ax.axis("off")
    for ax, (ds, bb) in zip(axes.ravel(), P_CELLS):
        for m, c in [("frozen", "k"), ("rls_feat", PCOL["rls"]), ("rls_bank_med", PCOL["bank"]),
                     (_bestgd(ds, bb, HON)[1], PCOL["gd"])]:
            row = PF_R[(PF_R.dataset == ds) & (PF_R.backbone == bb) & (PF_R.method == m)
                       & (PF_R.delay == HON) & (PF_R.seed == 0)]
            if not len(row): continue
            z = np.load(DIRS["results"] / f"{row.iloc[0].run_id}_curves.npz")
            ax.plot(z["curve_x"], z["curve"], color=c, lw=1.0, label=m)
        if (ds, bb) == ("weather", "dlinear"):
            try:
                _rf = PF_R[(PF_R.dataset == ds) & (PF_R.backbone == bb) & (PF_R.method == "frozen")
                           & (PF_R.delay == HON) & (PF_R.seed == 0)]
                _zf = np.load(DIRS["results"] / f"{_rf.iloc[0].run_id}_curves.npz")
                _cy, _cx = _zf["curve"], _zf["curve_x"]; _k = int(np.argmax(np.diff(_cy)))
                ax.annotate("regime shift", xy=(float(_cx[_k + 1]), float(_cy[_k + 1])), xytext=(0.42, 0.18),
                            textcoords="axes fraction", fontsize=6.5, arrowprops=dict(arrowstyle="->", lw=0.9))
            except Exception as _e:
                print("fig8 annotation skipped:", _e)
        ax.set_title(f"{ds}/{'DL' if bb=='dlinear' else 'PT'}"); ax.set_xlabel("online step")
    axes[0, 0].set_ylabel("cumulative MSE"); axes[0, 0].legend(frameon=False)
    fig.tight_layout(); _save(fig, "fig8_cumulative")

    fig, axes = plt.subplots(len(CFG.datasets), 2, figsize=(6.8, 2.1*len(CFG.datasets)), squeeze=False)
    for r, ds in enumerate(CFG.datasets):
        Dd = DATA[ds]; z = Dd["z"]; w = SEASONAL_PERIOD[ds]*7
        roll = pd.DataFrame(z).rolling(w, min_periods=w)
        for ax, stat, ttl in [(axes[r, 0], roll.mean(), "rolling mean (1w)"), (axes[r, 1], roll.std(), "rolling std (1w)")]:
            ax.plot(stat.to_numpy(), lw=0.4, alpha=0.7)
            for x0 in (Dd["n_train"], Dd["n_train"]+Dd["n_val"]): ax.axvline(x0, color="k", ls="--", lw=0.8)
            ax.set_title(f"{ds} — {ttl}")
    for ax in axes[-1, :]: ax.set_xlabel("$t$")
    fig.tight_layout(); _save(fig, "fig9_drift_overview")

print("paper figures complete ->", PAPER)

## Review round: bootstrap CIs for every external comparison present
Generic version of the external-CI cell: it discovers every `(dataset, seed, delay)` in `external_preds/`, pairs the bank
against each published method (default and `_tuned`), each method against the frozen model, and each `_tuned` variant
against its default, with the same circular moving-block bootstrap. Cached in `bootstrap_ci_external.json` (now with a
`seed` column). Safe to re-run after every new batch of external files.

In [ ]:
import re as _re2
ci_path = ROOT / "bootstrap_ci_external.json"
_rows = json.loads(ci_path.read_text()) if ci_path.exists() else []
for r in _rows: r.setdefault("seed", 0)
_have = {(r["dataset"], r["seed"], r["delay"], r["pair"]) for r in _rows}
_rng = np.random.default_rng(1)
_groups = {}
for p in sorted(EXT_DIR.glob("*.npz")):
    m = _re2.match(r"(.+?)__(\w+)__patchtst_ext__s(\d+)__d(\d+)\.npz", p.name)
    if m: _groups.setdefault((m.group(2), int(m.group(3)), int(m.group(4))), {})[m.group(1)] = p
def _ext_series(p):
    cp = SERIES_DIR / f"{cfg_hash({'ext': p.name, 'v': PROTO_V})}.npy"
    if cp.exists(): return np.load(cp)
    z = np.load(p, allow_pickle=False); P, meta, f = _align_to_ours(z)
    out = ((P - f["trues"]) ** 2).mean(axis=(1, 2)); np.save(cp, out); return out
def _ours_series(ds, seed, method, delay):
    p = EXT_DIR / f"frozen__{ds}__patchtst_ext__s{seed}__d24.npz"
    if not p.exists(): return None
    z = np.load(p, allow_pickle=False); P, meta, f = _align_to_ours(z)
    f2 = dict(f); f2["preds"] = P; T, C, H = P.shape
    return per_step_mse(f2, method, delay, C, H, ds=ds,
                        key={"ds": ds, "bb": "patchtst_ext", "seed": int(seed), "T": int(T), "v": PROTO_V})
for (ds, seed, d), files in sorted(_groups.items()):
    series = {}
    def get(name):
        if name in series: return series[name]
        if name in ("rls_bank_med", "naive", "frozen") or (name.startswith("elf_") and ds in CFG.elf_datasets):
            series[name] = _ours_series(ds, seed, name, d)
        else: series[name] = _ext_series(files[name]) if name in files else None
        return series[name]
    ext_methods = [m for m in files if m != "frozen"]
    pairs = [("rls_bank_med", m) for m in ext_methods] + [(m, "frozen") for m in ext_methods] + \
            [("rls_bank_med", "frozen"), ("rls_bank_med", "naive")] + \
            ([("elf_mix", m) for m in ext_methods] + [("elf_mix", "rls_bank_med"), ("elf_mix", "frozen"), ("elf_mix", "naive"),
              ("elf_linear", "rls_bank_med")] if ds in CFG.elf_datasets else []) + \
            [(m, m[:-6]) for m in ext_methods if m.endswith("_tuned") and m[:-6] in files]
    for a, b in pairs:
        key = (ds, seed, d, f"{a} - {b}")
        if key in _have: continue
        print(f"  ext CI {ds}/s{seed} d={d}: {a} - {b} ...", flush=True)
        sa, sb = get(a), get(b)
        if sa is None or sb is None: continue
        mean, lo, hi = mbb_ci(sa - sb, CFG.boot_block_mult * SEASONAL_PERIOD[ds], CFG.boot_B, _rng)
        _rows.append({"dataset": ds, "seed": seed, "delay": d, "pair": key[3], "mean_diff": round(mean, 4),
                      "ci_lo": round(lo, 4), "ci_hi": round(hi, 4), "significant": bool(lo > 0 or hi < 0)})
        _have.add(key); ci_path.write_text(json.dumps(_rows, indent=2))   # incremental

if _rows:
    E = pd.DataFrame(_rows).sort_values(["dataset", "seed", "delay", "pair"])
    print("negative mean_diff = first method better; * = 95% CI excludes 0 (their checkpoint, aligned stream)")
    print(E.assign(sig=np.where(E.significant, "*", "")).to_string(index=False))
else:
    print("no external prediction files found")